In [15]:
import mpmath
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import lu_factor, lu_solve, null_space
import bokeh.layouts
import bokeh.models
import bokeh.plotting

from IPython.display import display
from jupyter_bokeh.widgets import BokehModel
from bokeh.io import output_notebook, export_svg, export_png
from bokeh.plotting import figure
from ipywidgets import interact

import math

from bokeh.io import output_file, show, export_svgs
from bokeh.layouts import row
from bokeh.models import Ellipse, GraphRenderer, StaticLayoutProvider, Circle
from bokeh.palettes import Spectral6, Reds3, brewer
from bokeh.plotting import figure
from IPython.display import display
from bokeh.models import (Arrow, Label, NormalHead, OpenHead,
                          Plot, Range1d, TeeHead, VeeHead)
from bokeh.models import MathML

In [16]:
def solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma_1, gamma_2, alpha_14, alpha_41, dG_1=2, dG_2=2, k_bT=1):
    '''
    Solves for the steady-state probabilities of each state in the 6-state system.
    '''
    alpha_12 = alpha_21 * np.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * np.exp((dG_2 - epsilon_1)/ k_bT)

    alpha_54 = alpha_21
    alpha_65 = alpha_32
    alpha_45 = alpha_54 * np.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * np.exp((dG_2 - epsilon_2) / k_bT)



    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41

    a = np.array([[-(alpha_12 + alpha_14), alpha_21, gamma_1, alpha_41, 0 , 0],
                [alpha_12, -(alpha_21 + alpha_23 + alpha_25), alpha_32, 0, alpha_52, 0],
                [0, alpha_23, -(gamma_1 + alpha_32 + alpha_36), 0, 0, alpha_63],
                [alpha_14, 0, 0, -(alpha_41 + alpha_45), alpha_54, gamma_2],
                [0, alpha_25, 0, alpha_45, -(alpha_52 + alpha_54 + alpha_56), alpha_65],
                [0, 0, alpha_36, 0, alpha_56, -(gamma_2 + alpha_63 + alpha_65)]], dtype=float)  
    a[-1:,] = 1
    bnp = np.zeros(a.shape[0])
    bnp[-1] = 1

    a = mpmath.matrix(a.tolist())
    b = bnp.tolist()
    sol = np.array(mpmath.lu_solve(a,b)).astype(float)
    return sol

In [17]:




oscMidpoint = 0
k_bT = 1
dG_1 = 2
dG_2 = 2


alpha_21 = 1
alpha_32 = 1

alpha_54 = 1 
alpha_65 = 1 

N = 6
node_indices = list(range(N))

s1 = figure(title="Flux Diagram", x_range=(-1.1,1.1), y_range=(-1.1,1.1), tools="hover", tooltips="index: @name", toolbar_location=None)

graph = GraphRenderer()

graph.node_renderer.data_source.add(np.array(node_indices), 'index')
graph.node_renderer.data_source.add(np.array(node_indices) + 1, 'name')
graph.node_renderer.data_source.add(Spectral6, 'color')
graph.node_renderer.glyph = Ellipse(height=0.1, width=0.1, fill_color="color")

# s = [0, 2, 1, 1, 2, 3, 1, 2, 3, 3, 0, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 3, 0, 0, 1, 1, 2, 3, 4, 4, 5, 5, 0, 4, 2, 3, 4]
# s = [0, 2, 1, 1, 1, 2, 3, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 0, 1, 1, 4, 4, 5, 5, 0, 4, 2, 3, 4]

s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]

graph.edge_renderer.data_source.data = dict(
    start=s,
    end=e)

### start of layout code
x = [-0.5, 0, 0.5, -0.5, 0, 0.5]
y = [0.5, 0.5, 0.5, -0.5, -0.5, -0.5]
graph_layout = dict(zip(range(len(node_indices)), zip(x, y)))
graph.layout_provider = StaticLayoutProvider(graph_layout=graph_layout)


### Draw quadratic bezier paths
def bezier(start, end, control, steps):
    return [(1-s)**2*start + 2*(1-s)*s*control + s**2*end for s in steps]

xs, ys = [], []
steps = [i/100. for i in range(100)]
for edge_index in range(len(s)):
    sx, sy = graph_layout[s[edge_index]]
    ex, ey = graph_layout[e[edge_index]]
    match edge_index:
        # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
        # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
        case 0:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=15))
            s1.add_layout(arrow)
    
        case 1: 
            x = bezier(sx, ex, -0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 2:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.75, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 3:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 4:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 5:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 6:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 7:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 8:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 9: 
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.25, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 10:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 11:
            x = bezier(sx, ex, 0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 12:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.75, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 13:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 14:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, 1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 15:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, -1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=15))
            s1.add_layout(arrow)
        
        case _:
            xs.append(np.linspace(sx, ex, 100))
            ys.append(np.linspace(sy, ey, 100))
    

  
    
graph.edge_renderer.data_source.data['xs'] = xs
graph.edge_renderer.data_source.data['ys'] = ys



s1.renderers.append(graph)

probabilities = [r'$$p_1$$', r'$$p_2$$', r'$$p_3$$', r'$$p_4$$', r'$$p_5$$', r'$$p_6$$']
fluxes = [r'Path 1', r'Path 2', r'Path 1 + Path 2']

s2 = figure(title="State probabilities", x_range=probabilities, y_range=(0,1.0), tools="", toolbar_location=None)
s3 = figure(title="Flux", x_range=fluxes, y_range=(0,10), tools="", toolbar_location=None)
source_1 = bokeh.models.ColumnDataSource(data=dict(x=probabilities, top=[0,0,0,0, 0, 0], color=Spectral6))

source_2 = bokeh.models.ColumnDataSource(data=dict(x=fluxes, top=[0,0,0], color=Reds3))


s2.vbar(x='x', top='top', width=0.9, source=source_1, color='color')

s3.vbar(x='x', top='top', width=0.9, source=source_2, color='color')


 
def update(alpha_14=1, alpha_41=1, epsilon_amplitude=1, gamma=10, oscMidpoint=oscMidpoint):
    epsilon_1 = float(oscMidpoint + epsilon_amplitude/2)
    epsilon_2 = float(oscMidpoint - epsilon_amplitude/2)
    # alpha_41 = alpha_14
    p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=2, dG_2=2, k_bT=1)
    p1 = float(p1)
    p2 = float(p2)
    p3 = float(p3)
    p4 = float(p4)
    alpha_12 = alpha_21 * np.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * np.exp((dG_2 - epsilon_1)/ k_bT)


    alpha_45 = alpha_54 * np.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * np.exp((dG_2 - epsilon_2) / k_bT)

    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41
   
    source_1.data = dict(x=probabilities, top=[p1, p2, p3, p4, p5, p6], color=Spectral6)

    # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
    # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
    J_12 = np.max([p1 * alpha_12, 0.01])
    J_21 = np.max([p2 * alpha_21, 0.01])

    J_23 = np.max([p2 * alpha_23, 0.01])
    J_32 = np.max([p3 * alpha_32, 0.01])

    J_45 = np.max([p4 * alpha_45, 0.01])
    J_54 = np.max([p5 * alpha_54, 0.01])

    J_56 = np.max([p5 * alpha_56, 0.01])
    J_65 = np.max([p6 * alpha_65, 0.01])

    J_41 = np.max([p4 * alpha_41, 0.01])
    J_14 = np.max([p1 * alpha_14, 0.01])

    J_52 = np.max([p5 * alpha_52, 0.01])
    J_25 = np.max([p2 * alpha_25, 0.01])

    J_36 = np.max([p3 * alpha_36, 0.01])
    J_63 = np.max([p6 * alpha_63, 0.01])

    J_31 = np.max([p3 * gamma, 0.01])
    J_64 = np.max([p6 * gamma, 0.01])

    source_2.data = dict(x=fluxes, top=[J_31, J_64, J_31 + J_64], color=Reds3)



    graph.edge_renderer.data_source.data["line_width"] = np.array([J_12, J_14, J_23, J_25, J_36, J_45, J_56, J_21, J_41, J_32, J_52, J_63, J_54, J_65, J_31, J_64])
    # graph.edge_renderer.data_source.data["line_width"] = [1] * len(s)
    graph.edge_renderer.glyph.line_width = {'field': 'line_width'}

plot = row(s1, s2, s3)
interact(update, alpha_14=(0.001,500), alpha_41 =(0.001,500), epsilon_amplitude=(-5,5, 0.1), gamma=(0, 20, 0.1,), oscMidpoint=(-5,5,0.1))

display(BokehModel(plot))

interactive_python_plots = True
output_notebook()

interactive(children=(FloatSlider(value=1.0, description='alpha_14', max=500.0, min=0.001), FloatSlider(value=…

BokehModel(render_bundle={'docs_json': {'8011d7c0-0874-4fa2-9819-6607900ce98b': {'version': '3.6.2', 'title': …

Loading BokehJS ...

# Figure 1A,B:

In [18]:
def solve_p_dynamic_steady_state_eq(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma_1, gamma_2, alpha_14, alpha_41, dG_1=2, dG_2=2, k_bT=1):
    '''
    Solves for the steady-state probabilities of each state in the 6-state system.
    '''

    alpha_23 = alpha_32 * np.exp((dG_1-epsilon_1)/ k_bT)
    alpha_12 = alpha_21 * np.exp((dG_2 + epsilon_1) / k_bT)

    alpha_54 = alpha_21
    alpha_65 = alpha_32
    alpha_45 = alpha_54 * np.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * np.exp((dG_2 - epsilon_2) / k_bT)



    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41

    a = np.array([[-(alpha_12 + alpha_14 + gamma_1), alpha_21, gamma_1, alpha_41, 0 , 0],
                [alpha_12, -(alpha_21 + alpha_23 + alpha_25), alpha_32, 0, alpha_52, 0],
                [gamma_1, alpha_23, -(gamma_1 + alpha_32 + alpha_36), 0, 0, alpha_63],
                [alpha_14, 0, 0, -(alpha_41 + alpha_45 + gamma_2), alpha_54, gamma_2],
                [0, alpha_25, 0, alpha_45, -(alpha_52 + alpha_54 + alpha_56), alpha_65],
                [0, 0, alpha_36, gamma_2, alpha_56, -(gamma_2 + alpha_63 + alpha_65)]], dtype=float)  
    a[-1:,] = 1
    bnp = np.zeros(a.shape[0])
    bnp[-1] = 1

    a = mpmath.matrix(a.tolist())
    b = bnp.tolist()
    sol = np.array(mpmath.lu_solve(a,b)).astype(float)
    return sol

In [19]:

oscMidpoint = 0
k_bT = 1
dG_1 = 0
dG_2 = 0
epsilon_amplitude = 0
gamma=10

alpha_21 = 1
alpha_32 = 1

alpha_54 = 1 
alpha_65 = 1 

N = 6
node_indices = list(range(N))

s1 = figure(title="Flux Diagram", x_range=(-1.1,1.1), y_range=(0.2,1.8), tools="hover", tooltips="index: @name", toolbar_location=None, width=208, height=208)
s1.axis.visible = False

graph = GraphRenderer()

graph.node_renderer.data_source.add(np.array(node_indices), 'index')
graph.node_renderer.data_source.add(np.array(node_indices) + 1, 'name')
graph.node_renderer.data_source.add(Spectral6, 'color')
graph.node_renderer.glyph = Circle(radius=0.1, fill_color="color")

# s = [0, 2, 1, 1, 2, 3, 1, 2, 3, 3, 0, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 3, 0, 0, 1, 1, 2, 3, 4, 4, 5, 5, 0, 4, 2, 3, 4]
# s = [0, 2, 1, 1, 1, 2, 3, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 0, 1, 1, 4, 4, 5, 5, 0, 4, 2, 3, 4]

s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5, 0]
e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3, 2]

graph.edge_renderer.data_source.data = dict(
    start=s,
    end=e)

### start of layout code
x = [-0.5, 0, 0.5, -0.5, 0, 0.5]
y = np.array([0.5, 0.5, 0.5, -0.5, -0.5, -0.5]) + 0.5
graph_layout = dict(zip(range(len(node_indices)), zip(x, y)))
graph.layout_provider = StaticLayoutProvider(graph_layout=graph_layout)


### Draw quadratic bezier paths
def bezier(start, end, control, steps):
    return [(1-s)**2*start + 2*(1-s)*s*control + s**2*end for s in steps]
vSize=6
xs, ys = [], []
steps = [i/100. for i in range(100)]
for edge_index in range(len(s)):
    sx, sy = graph_layout[s[edge_index]]
    ex, ey = graph_layout[e[edge_index]]
    match edge_index:
        # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
        # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
        case 0:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 1.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
    
        case 1: 
            x = bezier(sx, ex, -0.75, steps)
            y = bezier(sy, ey, 0.5, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 2:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 1.25, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 3:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.5, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 4:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.5, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 5:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 6:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 7:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 8:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.5, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            # s1.add_layout(arrow)
        case 9: 
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.75, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 10:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.5, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            # s1.add_layout(arrow)
        case 11:
            x = bezier(sx, ex, 0.75, steps)
            y = bezier(sy, ey, 0.5, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            # s1.add_layout(arrow)
        case 12:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.25, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 13:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 14:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, 2.0, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 15:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, -1.0, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            # s1.add_layout(arrow)
        case 16:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, 0, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        
        case _:
            xs.append(np.linspace(sx, ex, 100))
            ys.append(np.linspace(sy, ey, 100))
    

  
    
graph.edge_renderer.data_source.data['xs'] = xs
graph.edge_renderer.data_source.data['ys'] = ys



s1.renderers.append(graph)

probabilities = [r'1', r'2', r'3']

s2 = figure(title="State probabilities", x_range=probabilities, y_range=(0,1.0), tools="", toolbar_location=None)
s2.xaxis.axis_label = 'State'
s2.yaxis.axis_label = 'Probability'
source_1 = bokeh.models.ColumnDataSource(data=dict(x=probabilities, top=[0,0,0], color=Spectral6[:3]))

s2.vbar(x='x', top='top', width=0.9, source=source_1, color='color')


s3 = figure(title="Reaction coordinate", x_range=(-1.1,1.1), y_range=(-10, 10), tools="", toolbar_location=None, width=208, height=208)
s3.xaxis.axis_label = 'Reaction Coordinate'
s3.yaxis.axis_label = '$$\Delta G \ (k_B T)$$'

n_states = 3
rxn_coord_indices = list(range(n_states * 2))

x_rxncoord = np.linspace(-0.9, 0.9, len(rxn_coord_indices))
y_rxncoord = [0] * len(rxn_coord_indices)
s_rxn_coord = rxn_coord_indices[:-1]
e_rxn_coord = rxn_coord_indices[1:]

graph_rxn_coord = GraphRenderer()
graph_rxn_coord.node_renderer.data_source.add(np.array(rxn_coord_indices), 'index')
graph_rxn_coord.edge_renderer.data_source.data = dict(
    start=s_rxn_coord,
    end=e_rxn_coord
)
### start of layout code

graph_layout_rxncoord = dict(zip(range(len(rxn_coord_indices)), zip(x_rxncoord, y_rxncoord)))

graph_rxn_coord.node_renderer.glyph = Ellipse(height=0.0, width=0.0)

graph_rxn_coord.layout_provider = StaticLayoutProvider(graph_layout=graph_layout_rxncoord)


def set_rxn_coord(s, e ):
    xs, ys = [], []
    for edge_index in range(len(s)):
        sx, sy = graph_layout_rxncoord[s[edge_index]]
        ex, ey = graph_layout_rxncoord[e[edge_index]]

        xs.append(np.linspace(sx, ex, 100))
        ys.append(np.linspace(sy, ey, 100))
    return xs, ys

xs, ys = set_rxn_coord(s_rxn_coord, e_rxn_coord)
graph_rxn_coord.edge_renderer.data_source.data['xs'] = xs
graph_rxn_coord.edge_renderer.data_source.data['ys'] = ys

s3.renderers.append(graph_rxn_coord)





 
def update(dG_1 = 2, dG_2 = 2):
    alpha_14 = 1e11
    alpha_41 = 1e11
    epsilon_1 = float(oscMidpoint + epsilon_amplitude/2)
    epsilon_2 = float(oscMidpoint - epsilon_amplitude/2)
    # alpha_41 = alpha_14
    p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state_eq(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    p1 = float(p1)
    p2 = float(p2)
    p3 = float(p3)
    p4 = float(p4)
    p5 = float(p5)
    p6 = float(p6)
    alpha_12 = alpha_21 * np.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * np.exp((dG_2 - epsilon_1)/ k_bT)


    alpha_45 = alpha_54 * np.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * np.exp((dG_2 - epsilon_2) / k_bT)

    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41
   

    # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
    # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
    J_12 = np.max([p1 * alpha_12, 0.01])
    J_21 = np.max([p2 * alpha_21, 0.01])

    J_23 = np.max([p2 * alpha_23, 0.01])
    J_32 = np.max([p3 * alpha_32, 0.01])

    J_45 = np.max([p4 * alpha_45, 0.01])
    J_54 = np.max([p5 * alpha_54, 0.01])

    J_56 = np.max([p5 * alpha_56, 0.01])
    J_65 = np.max([p6 * alpha_65, 0.01])

    J_41 = np.max([p4 * alpha_41, 0.01])
    J_14 = np.max([p1 * alpha_14, 0.01])

    J_52 = np.max([p5 * alpha_52, 0.01])
    J_25 = np.max([p2 * alpha_25, 0.01])

    J_36 = np.max([p3 * alpha_36, 0.01])
    J_63 = np.max([p6 * alpha_63, 0.01])

    J_31 = np.max([p3 * gamma, 0.01])
    J_13 = np.max([p1 * gamma, 0.01])
    J_64 = np.max([p6 * gamma, 0.01])

    x_rxncoord = np.linspace(-0.9, 0.9, len(rxn_coord_indices))
    y_rxncoord = [0, 0, -dG_1, -dG_1, -(dG_2 + dG_1), -(dG_2 + dG_1)]

    xs, ys = [], []
    for edge_index in range(len(x_rxncoord) - 1):
        sx, sy = x_rxncoord[edge_index], y_rxncoord[edge_index]
        ex, ey = x_rxncoord[edge_index + 1], y_rxncoord[edge_index + 1]

        xs.append(np.linspace(sx, ex, 100))
        ys.append(np.linspace(sy, ey, 100))
        
    s3.renderers[0].edge_renderer.data_source.data['xs'] = xs
    s3.renderers[0].edge_renderer.data_source.data['ys'] = ys
    s3.renderers[0].edge_renderer.data_source.data["line_dash"] = ['solid', 'dashed', 'solid', 'dashed', 'solid']
    graph_rxn_coord.edge_renderer.glyph.line_dash = {'field': 'line_dash'}
    graph_rxn_coord.edge_renderer.data_source.data["line_width"] = [2] * len(s_rxn_coord)
    graph_rxn_coord.edge_renderer.glyph.line_width = {'field': 'line_width'}



    source_1.data = dict(x=probabilities, top=[p1 + p4, p2 + p5, p3 + p6], color=Spectral6[:3])
    blank = 1e-11
    graph.edge_renderer.data_source.data["line_width"] = np.array([J_12, blank, J_23, blank, blank, J_45, J_56, J_21, blank, J_32, blank, blank, J_54, J_65, J_31, J_64, J_13])
    # graph.edge_renderer.data_source.data["line_width"] = [1] * len(s)
    graph.edge_renderer.glyph.line_width = {'field': 'line_width'}

    # graph_rxn_coord.edge_renderer.data_source.data["line_width"] = [1] * len(s_rxn_coord)
    # # graph.edge_renderer.data_source.data["line_width"] = [1] * len(s)
    # graph_rxn_coord.edge_renderer.glyph.line_width = {'field': 'line_width'}
# plot = row(s1,s3, s2)


plot = row(s1,s3)
interact(update, dG_1 = (-5,5,0.1), dG_2 = (-5,5,0.1))

display(BokehModel(plot))
# # export_png(plot, filename="equilibrium_single.png")
# s1.output_backend = "svg"
# s2.output_backend = "svg"
# s3.output_backend = "svg"
# export_svg(plot, filename="equilibrium_single.svg")

interactive_python_plots = True
output_notebook()

<>:205: SyntaxWarning: invalid escape sequence '\D'
<>:205: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_57362/1673585643.py:205: SyntaxWarning: invalid escape sequence '\D'
  s3.yaxis.axis_label = '$$\Delta G \ (k_B T)$$'


interactive(children=(FloatSlider(value=2.0, description='dG_1', max=5.0, min=-5.0), FloatSlider(value=2.0, de…

BokehModel(render_bundle={'docs_json': {'c894cacc-88ce-4105-a1ef-613f16c676d0': {'version': '3.6.2', 'title': …

Loading BokehJS ...

# Fig. 1C,D



In [20]:
oscMidpoint = 0
k_bT = 1
dG_1 = 2
dG_2 = 2
epsilon = 0

alpha_21 = 1
alpha_32 = 1

alpha_54 = 1 
alpha_65 = 1 

N = 6
node_indices = list(range(N))

s1 = figure(title="Flux Diagram", x_range=(-1.1,1.1), y_range=(0.2,1.8), tools="hover", tooltips="index: @name", toolbar_location=None, width=208, height=208)
s1.axis.visible = False

graph = GraphRenderer()

graph.node_renderer.data_source.add(np.array(node_indices), 'index')
graph.node_renderer.data_source.add(np.array(node_indices) + 1, 'name')
graph.node_renderer.data_source.add(Spectral6, 'color')
graph.node_renderer.glyph = Circle(radius=0.1, fill_color="color")

# s = [0, 2, 1, 1, 2, 3, 1, 2, 3, 3, 0, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 3, 0, 0, 1, 1, 2, 3, 4, 4, 5, 5, 0, 4, 2, 3, 4]
# s = [0, 2, 1, 1, 1, 2, 3, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 0, 1, 1, 4, 4, 5, 5, 0, 4, 2, 3, 4]

s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]

graph.edge_renderer.data_source.data = dict(
    start=s,
    end=e)

### start of layout code
x = [-0.5, 0, 0.5, -0.5, 0, 0.5]
y = np.array([0.5, 0.5, 0.5, -0.5, -0.5, -0.5]) + 0.5
graph_layout = dict(zip(range(len(node_indices)), zip(x, y)))
graph.layout_provider = StaticLayoutProvider(graph_layout=graph_layout)


### Draw quadratic bezier paths
def bezier(start, end, control, steps):
    return [(1-s)**2*start + 2*(1-s)*s*control + s**2*end for s in steps]
vSize=6
xs, ys = [], []
steps = [i/100. for i in range(100)]
for edge_index in range(len(s)):
    sx, sy = graph_layout[s[edge_index]]
    ex, ey = graph_layout[e[edge_index]]
    match edge_index:
        # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
        # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
        case 0:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 1.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
    
        case 1: 
            x = bezier(sx, ex, -0.75, steps)
            y = bezier(sy, ey, 0.5, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 2:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 1.25, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 3:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.5, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 4:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.5, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 5:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 6:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 7:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 8:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.5, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            # s1.add_layout(arrow)
        case 9: 
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.75, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 10:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.5, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            # s1.add_layout(arrow)
        case 11:
            x = bezier(sx, ex, 0.75, steps)
            y = bezier(sy, ey, 0.5, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            # s1.add_layout(arrow)
        case 12:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.25, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 13:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 14:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, 2.0, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 15:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, -1.0, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)

        case _:
            xs.append(np.linspace(sx, ex, 100))
            ys.append(np.linspace(sy, ey, 100))
    

  
    
graph.edge_renderer.data_source.data['xs'] = xs
graph.edge_renderer.data_source.data['ys'] = ys



s1.renderers.append(graph)

probabilities = [r'$$p_1$$', r'$$p_2$$', r'$$p_3$$']
# probabilities = [r'$$p_1$$', r'$$p_2$$', r'$$p_3$$', r'$$p_4$$', r'$$p_5$$', r'$$p_6$$']
fluxes = [r'Path 1', r'Path 2', r'Path 1 + Path 2']

s2 = figure(title="State probabilities", x_range=probabilities, y_range=(0,1.0), tools="", toolbar_location=None, width=208, height=208)
source_1 = bokeh.models.ColumnDataSource(data=dict(x=probabilities, top=[0,0,0], color=Spectral6[:3]))
# source_1 = bokeh.models.ColumnDataSource(data=dict(x=probabilities, top=[0,0,0, 0, 0,0], color=Spectral6))

s2.vbar(x='x', top='top', width=0.9, source=source_1, color='color')

s3 = figure(title="Flux", x_range=(-5, 5), y_range=(0,5), tools="", toolbar_location=None, width=208, height=208)

epsilon_list = np.linspace(-5,5,50)
J_tot_list = []
for epsilon in epsilon_list:
    epsilon_1 = epsilon
    epsilon_2 = epsilon
    alpha_14 = 1e-11
    alpha_41 = 1e-11
    # alpha_41 = alpha_14
    p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    p1 = float(p1)
    p2 = float(p2)
    p3 = float(p3)
    p4 = float(p4)
    p5 = float(p5)
    p6 = float(p6)

   
    J_31 = np.max([p3 * gamma, 0.01])
    J_64 = np.max([p6 * gamma, 0.01])

    J_tot = J_31 + J_64
    J_tot_list.append(J_tot)


source_2 = bokeh.models.ColumnDataSource(data=dict(x=epsilon_list, y=J_tot_list))
idx = np.argmin(np.abs(epsilon_list - epsilon))


intersect = s3.circle([epsilon_list[idx]], [J_tot_list[idx]], size=8, color='red')
s3.line(x='x', y='y', source=source_2, line_width=2, color='black')
s3.xaxis.axis_label = '$$\\epsilon \ (k_B T)$$'
s3.yaxis.axis_label = '$$J_{tot}$$ [$$1/s$$]'
 
def update(epsilon=0, gamma=10, dG_1=2, dG_2=2, alpha_21=1, alpha_32=1):
    idx = np.argmin(np.abs(epsilon_list - epsilon))
    alpha_54 = alpha_21
    alpha_65 = alpha_32
    epsilon_1 = epsilon
    epsilon_2 = epsilon
    alpha_14 = 1e11
    alpha_41 = 1e11 
    # alpha_41 = alpha_14
    p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    p1 = float(p1)
    p2 = float(p2)
    p3 = float(p3)
    p4 = float(p4)
    p5 = float(p5)
    p6 = float(p6)
    
    alpha_12 = alpha_21 * np.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * np.exp((dG_2 - epsilon_1)/ k_bT)


    alpha_45 = alpha_54 * np.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * np.exp((dG_2 - epsilon_2) / k_bT)

    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41
   
    source_1.data = dict(x=probabilities, top=[p1 + p4, p2 + p5, p3 + p6], color=Spectral6[:3])
    # source_1.data = dict(x=probabilities, top=[p1, p2, p3,  p4, p5, p6], color=Spectral6)

    # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
    # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
    J_12 = np.max([p1 * alpha_12, 0.01])
    J_21 = np.max([p2 * alpha_21, 0.01])

    J_23 = np.max([p2 * alpha_23, 0.01])
    J_32 = np.max([p3 * alpha_32, 0.01])

    J_45 = np.max([p4 * alpha_45, 0.01])
    J_54 = np.max([p5 * alpha_54, 0.01])

    J_56 = np.max([p5 * alpha_56, 0.01])
    J_65 = np.max([p6 * alpha_65, 0.01])

    J_41 = np.max([p4 * alpha_41, 0.01])
    J_14 = np.max([p1 * alpha_14, 0.01])

    J_52 = np.max([p5 * alpha_52, 0.01])
    J_25 = np.max([p2 * alpha_25, 0.01])

    J_36 = np.max([p3 * alpha_36, 0.01])
    J_63 = np.max([p6 * alpha_63, 0.01])

    J_31 = np.max([p3 * gamma, 0.01])
    J_64 = np.max([p6 * gamma, 0.01])

    blank = 1e-11
    graph.edge_renderer.data_source.data["line_width"] = np.array([J_12, blank, J_23, blank, blank, J_45, J_56, J_21, blank, J_32, blank, blank, J_54, J_65, J_31, J_64])
    # graph.edge_renderer.data_source.data["line_width"] = [1] * len(s)
    graph.edge_renderer.glyph.line_width = {'field': 'line_width'}


    J_tot_list = []
    for epsilon in epsilon_list:
        epsilon_1 = epsilon
        epsilon_2 = epsilon
        p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
        p1 = float(p1)
        p2 = float(p2)
        p3 = float(p3)
        p4 = float(p4)
        p5 = float(p5)
        p6 = float(p6)

        J_31 = np.max([p3 * gamma, 0.01])
        J_64 = np.max([p6 * gamma, 0.01])

        J_tot = J_31 + J_64
        J_tot_list.append(J_tot)


    source_2.data = dict(x=epsilon_list, y=J_tot_list)
    intersect.data_source.data['y'] = [J_tot_list[idx]]
    intersect.data_source.data['x'] = [epsilon_list[idx]]
    
    
# plotting new graph

# plot = row(s1, s3, s2)
plot = row(s1, s3)
interact(update, epsilon=(-5,5, 0.1), gamma=(0, 20, 0.1,), dG_1 = (-5,5,0.1), dG_2 = (-5,5,0.1), alpha_21 = (0.1,100,0.1), alpha_32 = (0.1,100,0.1))
# s1.output_backend = "svg"
# s2.output_backend = "svg"
# s3.output_backend = "svg"
# export_svg(plot, filename="steady_state_single.svg")

display(BokehModel(plot))


<>:227: SyntaxWarning: invalid escape sequence '\ '
<>:227: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_57362/2805134487.py:227: SyntaxWarning: invalid escape sequence '\ '
  s3.xaxis.axis_label = '$$\\epsilon \ (k_B T)$$'


interactive(children=(FloatSlider(value=0.0, description='epsilon', max=5.0, min=-5.0), FloatSlider(value=10.0…

BokehModel(render_bundle={'docs_json': {'232fd2f4-43f7-4446-8ae4-e8b7545f60ff': {'version': '3.6.2', 'title': …

# Figure 2A,B

In [21]:
def heatmap(A, F, J_turnover_total_list, alpha_21, alpha_32, gamma, alpha_14_fraction, dG_1, dG_2, oscMidpoint):
    turnover_fold_increase = np.zeros(A.shape)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):

            epsilon_1 = float(oscMidpoint + A[i, j])
            epsilon_2 = float(oscMidpoint - A[i, j])

            alpha_14 = F[i, j] * alpha_14_fraction
            alpha_41 = F[i, j] * (1 - alpha_14_fraction)

            p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
            flux = p3 * gamma + p6 * gamma
            turnover_fold_increase[i, j] = flux / max(J_turnover_total_list)
    return turnover_fold_increase

def find_J_tot_list(epsilon_list, dG_1, dG_2, alpha_21, alpha_32, gamma):
    J_tot_list = []
    alpha_14 = 1e11
    alpha_41 = 1e11 
    for epsilon in epsilon_list:
        epsilon_1 = epsilon
        epsilon_2 = epsilon
        p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
        p1 = float(p1)
        p2 = float(p2)
        p3 = float(p3)
        p4 = float(p4)
        p5 = float(p5)
        p6 = float(p6)

        J_31 = np.max([p3 * gamma, 0.01])
        J_64 = np.max([p6 * gamma, 0.01])

        J_tot = J_31 + J_64
        J_tot_list.append(J_tot)
    return J_tot_list

In [22]:
from bokeh.models import LinearColorMapper, ColorBar, BasicTicker
oscMidpoint = 0
k_bT = 1
dG_1 = 2
dG_2 = 2


alpha_21 = 1
alpha_32 = 1

alpha_54 = 1 
alpha_65 = 1 


## S1 definition
s1 = figure(title="Flux Diagram", x_range=(-1.1,1.1), y_range=(-1.1,1.1), tools="hover", tooltips="index: @name", toolbar_location=None)
s1.axis.visible = False
N = 6
node_indices = list(range(N))
graph = GraphRenderer()

graph.node_renderer.data_source.add(np.array(node_indices), 'index')
graph.node_renderer.data_source.add(np.array(node_indices) + 1, 'name')
graph.node_renderer.data_source.add(Spectral6, 'color')
graph.node_renderer.glyph = Ellipse(height=0.1, width=0.1, fill_color="color")

# s = [0, 2, 1, 1, 2, 3, 1, 2, 3, 3, 0, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 3, 0, 0, 1, 1, 2, 3, 4, 4, 5, 5, 0, 4, 2, 3, 4]
# s = [0, 2, 1, 1, 1, 2, 3, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 0, 1, 1, 4, 4, 5, 5, 0, 4, 2, 3, 4]

s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]

graph.edge_renderer.data_source.data = dict(
    start=s,
    end=e)

### start of layout code
x = [-0.5, 0, 0.5, -0.5, 0, 0.5]
y = [0.5, 0.5, 0.5, -0.5, -0.5, -0.5]
graph_layout = dict(zip(range(len(node_indices)), zip(x, y)))
graph.layout_provider = StaticLayoutProvider(graph_layout=graph_layout)


### Draw quadratic bezier paths
def bezier(start, end, control, steps):
    return [(1-s)**2*start + 2*(1-s)*s*control + s**2*end for s in steps]

xs, ys = [], []
steps = [i/100. for i in range(100)]
for edge_index in range(len(s)):
    sx, sy = graph_layout[s[edge_index]]
    ex, ey = graph_layout[e[edge_index]]
    match edge_index:
        # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
        # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
        case 0:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=15))
            s1.add_layout(arrow)
    
        case 1: 
            x = bezier(sx, ex, -0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 2:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.75, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 3:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 4:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 5:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 6:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 7:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 8:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 9: 
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.25, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 10:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 11:
            x = bezier(sx, ex, 0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 12:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.75, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 13:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 14:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, 1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=15))
            s1.add_layout(arrow)
        case 15:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, -1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=15))
            s1.add_layout(arrow)
        
        case _:
            xs.append(np.linspace(sx, ex, 100))
            ys.append(np.linspace(sy, ey, 100))
    

  
    
graph.edge_renderer.data_source.data['xs'] = xs
graph.edge_renderer.data_source.data['ys'] = ys



s1.renderers.append(graph)

## S2 definition
probabilities = [r'1', r'2', r'3', r'4', r'5', r'6']
s2 = figure(title="State probabilities", x_range=probabilities, y_range=(0,1.0), tools="", toolbar_location=None)
source_1 = bokeh.models.ColumnDataSource(data=dict(x=probabilities, top=[0,0,0,0, 0, 0], color=Spectral6))

s2.vbar(x='x', top='top', width=0.9, source=source_1, color='color')

## S3 definition
max_amplitude = 5
max_frequency = 500
s3 = figure(title="Flux Heatmap", x_range=(0,max_amplitude), y_range=(0, max_frequency), tools="",  toolbar_location=None)
n_points = 20

alpha_14_fraction = 0.5

amplitudes = np.linspace(0, max_amplitude, n_points)
frequencies = np.linspace(1e-5, max_frequency, n_points)
epsilon_list = np.linspace(-5,5,50)

amplitude = 2
frequency = 250

A, F = np.meshgrid(amplitudes, frequencies)
J_tot_list = find_J_tot_list(epsilon_list, dG_1, dG_2, alpha_21, alpha_32, gamma)
turnover_fold_increase = heatmap(A, F, J_tot_list, alpha_21, alpha_32, gamma, alpha_14_fraction, dG_1, dG_2, oscMidpoint)
color_mapper = LinearColorMapper(palette="Viridis256", low=0, high=np.max(turnover_fold_increase))

img = s3.image(image=[turnover_fold_increase], x=0, y=0, dw=max_amplitude, dh=max_frequency, color_mapper=color_mapper)
s3.xaxis.axis_label = r'Oscillation Amplitude $$|\epsilon| \ [k_B T]$$'
s3.yaxis.axis_label = r'Oscillation Timescale $$ \alpha_{I,II}+\alpha_{II,I}\ [(s^{-1})]$$'

color_bar = ColorBar(color_mapper=color_mapper, ticker=BasicTicker(),
                     label_standoff=12, border_line_color=None, location=(0,0), title="Turnover Fold Increase over Static Maximum")

s3.add_layout(color_bar, 'right')

idx_X = np.argmin(np.abs(amplitudes - amplitude))
idx_Y = np.argmin(np.abs(frequencies - frequency))
intersect = s3.circle([amplitudes[idx_X]], [frequencies[idx_Y]], size=8, color='red')

 
def update(alpha_14_fraction=0.5, gamma=10, oscMidpoint=oscMidpoint, dG_1=2, dG_2=2, amplitude=2, alpha_tr_total=250):
    #s1 update
    epsilon_1 = float(oscMidpoint + amplitude/2)
    epsilon_2 = float(oscMidpoint - amplitude/2)

    alpha_14 = alpha_tr_total * alpha_14_fraction
    alpha_41 = alpha_tr_total * (1 - alpha_14_fraction)
    
    # alpha_41 = alpha_14
    p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    p1 = float(p1)
    p2 = float(p2)
    p3 = float(p3)
    p4 = float(p4)
    alpha_12 = alpha_21 * np.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * np.exp((dG_2 - epsilon_1)/ k_bT)


    alpha_45 = alpha_54 * np.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * np.exp((dG_2 - epsilon_2) / k_bT)

    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41


    # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
    # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
    J_12 = np.max([p1 * alpha_12, 0.01])
    J_21 = np.max([p2 * alpha_21, 0.01])

    J_23 = np.max([p2 * alpha_23, 0.01])
    J_32 = np.max([p3 * alpha_32, 0.01])

    J_45 = np.max([p4 * alpha_45, 0.01])
    J_54 = np.max([p5 * alpha_54, 0.01])

    J_56 = np.max([p5 * alpha_56, 0.01])
    J_65 = np.max([p6 * alpha_65, 0.01])

    J_41 = np.max([p4 * alpha_41, 0.01])
    J_14 = np.max([p1 * alpha_14, 0.01])

    J_52 = np.max([p5 * alpha_52, 0.01])
    J_25 = np.max([p2 * alpha_25, 0.01])

    J_36 = np.max([p3 * alpha_36, 0.01])
    J_63 = np.max([p6 * alpha_63, 0.01])

    J_31 = np.max([p3 * gamma, 0.01])
    J_64 = np.max([p6 * gamma, 0.01])




    graph.edge_renderer.data_source.data["line_width"] = np.array([J_12, J_14, J_23, J_25, J_36, J_45, J_56, J_21, J_41, J_32, J_52, J_63, J_54, J_65, J_31, J_64])
    # graph.edge_renderer.data_source.data["line_width"] = [1] * len(s)
    graph.edge_renderer.glyph.line_width = {'field': 'line_width'}
    # S2 update 
    source_1.data = dict(x=probabilities, top=[p1, p2, p3, p4, p5, p6], color=Spectral6)

    # S3 update
    J_tot_list = find_J_tot_list(epsilon_list, dG_1, dG_2, alpha_21, alpha_32, gamma)
    turnover_fold_increase = heatmap(A, F, J_tot_list, alpha_21, alpha_32, gamma, alpha_14_fraction, dG_1, dG_2, oscMidpoint)
    color_mapper.high = np.max(turnover_fold_increase)
    img.data_source.data['image'] = [turnover_fold_increase]
    idx_X = np.argmin(np.abs(amplitudes - amplitude))
    idx_Y = np.argmin(np.abs(frequencies - alpha_tr_total))
    intersect.data_source.data['y'] = [frequencies[idx_Y]]
    intersect.data_source.data['x'] = [amplitudes[idx_X]]

plot = row(s1, s3, s2)
interact(update, alpha_14_fraction =(0,1, 0.001), gamma=(0, 20, 0.1,), oscMidpoint=(-5,5,0.1), dG_1 = (-5,5,0.1), dG_2 = (-5,5,0.1), amplitude=(0,5,0.1), alpha_tr_total=(1e-11,500,1))

display(BokehModel(plot))

interactive_python_plots = True
output_notebook()

interactive(children=(FloatSlider(value=0.5, description='alpha_14_fraction', max=1.0, step=0.001), FloatSlide…

BokehModel(render_bundle={'docs_json': {'22e62ace-d968-49a6-890c-0a7ad3e67542': {'version': '3.6.2', 'title': …

Loading BokehJS ...

# Fig 2A,B redo

In [23]:
def heatmap(F1, F2, J_turnover_total_list, alpha_21, alpha_32, gamma, dG_1, dG_2, oscMidpoint, amplitude):
    turnover_fold_increase = np.zeros(F1.shape)
    for i in range(F1.shape[0]):
        for j in range(F1.shape[1]):

            epsilon_1 = float(oscMidpoint + amplitude)
            epsilon_2 = float(oscMidpoint - amplitude)

            alpha_14 = F1[i, j]
            alpha_41 = F2[i, j] 

            p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
            flux = p3 * gamma + p6 * gamma
            turnover_fold_increase[i, j] = flux / max(J_turnover_total_list)
    return turnover_fold_increase

In [24]:
from bokeh.models import LinearColorMapper, ColorBar, BasicTicker, LogColorMapper

palette = brewer['PuOr'][11][::-1]
palette = bokeh.palettes.interp_palette(palette, 400)
oscMidpoint = 0
k_bT = 1
dG_1 = 2
dG_2 = 2


alpha_21 = 1
alpha_32 = 1

alpha_54 = 1 
alpha_65 = 1 


## S1 definition
s1 = figure(title="Flux Diagram", x_range=(-1.3,1.3), y_range=(-1.05,1.05), tools="hover", tooltips="index: @name", toolbar_location=None, width=208, height=208)
s1.axis.visible = False
N = 6
node_indices = list(range(N))
graph = GraphRenderer()

graph.node_renderer.data_source.add(np.array(node_indices), 'index')
graph.node_renderer.data_source.add(np.array(node_indices) + 1, 'name')
graph.node_renderer.data_source.add(Spectral6, 'color')
graph.node_renderer.glyph = Circle(radius=0.1,fill_color="color")

# s = [0, 2, 1, 1, 2, 3, 1, 2, 3, 3, 0, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 3, 0, 0, 1, 1, 2, 3, 4, 4, 5, 5, 0, 4, 2, 3, 4]
# s = [0, 2, 1, 1, 1, 2, 3, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 0, 1, 1, 4, 4, 5, 5, 0, 4, 2, 3, 4]

s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]

graph.edge_renderer.data_source.data = dict(
    start=s,
    end=e)

### start of layout code
x = [-0.5, 0, 0.5, -0.5, 0, 0.5]
y = [0.5, 0.5, 0.5, -0.5, -0.5, -0.5]
graph_layout = dict(zip(range(len(node_indices)), zip(x, y)))
graph.layout_provider = StaticLayoutProvider(graph_layout=graph_layout)


### Draw quadratic bezier paths
def bezier(start, end, control, steps):
    return [(1-s)**2*start + 2*(1-s)*s*control + s**2*end for s in steps]
vSize = 6
xs, ys = [], []
steps = [i/100. for i in range(100)]
for edge_index in range(len(s)):
    sx, sy = graph_layout[s[edge_index]]
    ex, ey = graph_layout[e[edge_index]]
    match edge_index:
        # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
        # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
        case 0:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
    
        case 1: 
            x = bezier(sx, ex, -0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 2:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.75, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 3:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 4:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 5:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 6:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 7:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 8:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 9: 
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.25, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 10:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 11:
            x = bezier(sx, ex, 0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 12:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.75, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 13:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 14:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, 1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 15:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, -1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        
        case _:
            xs.append(np.linspace(sx, ex, 100))
            ys.append(np.linspace(sy, ey, 100))
    

  
    
graph.edge_renderer.data_source.data['xs'] = xs
graph.edge_renderer.data_source.data['ys'] = ys



s1.renderers.append(graph)

## S2 definition
probabilities = [r'1', r'2', r'3', r'4', r'5', r'6']
s2 = figure(title="State probabilities", x_range=probabilities, y_range=(0,1.0), tools="", toolbar_location=None)
source_1 = bokeh.models.ColumnDataSource(data=dict(x=probabilities, top=[0,0,0,0, 0, 0], color=Spectral6))

s2.vbar(x='x', top='top', width=0.9, source=source_1, color='color')

## S3 definition
max_frequency = 100

s3 = figure(title="Flux Heatmap", x_range=(1e-5,max_frequency), y_range=(1e-5, max_frequency), tools="",  toolbar_location=None, width=278, height=208)
n_points = 20
alpha_14_fraction = 0.5

frequencies_1 = np.linspace(1e-5, max_frequency, n_points)
frequencies_2 = np.linspace(1e-5, max_frequency, n_points)
epsilon_list = np.linspace(-5,5,50)

amplitude = 2
frequency = 250

F1, F2 = np.meshgrid(frequencies_1, frequencies_2)
J_tot_list = find_J_tot_list(epsilon_list, dG_1, dG_2, alpha_21, alpha_32, gamma)
turnover_fold_increase = heatmap(F1, F2, J_tot_list, alpha_21, alpha_32, gamma, dG_1, dG_2, oscMidpoint, amplitude)
bound = np.max([np.abs(np.max(np.log10(turnover_fold_increase))), np.abs(np.min(np.log10(turnover_fold_increase)))])
color_mapper = LogColorMapper(palette=palette, low=10**-bound, high=10**bound)


img = s3.image(image=[np.log10(turnover_fold_increase)], x=0, y=0, dw=max_frequency, dh=max_frequency, color_mapper=color_mapper)
s3.xaxis.axis_label = r'$$ \alpha_{I,II}\ [(s^{-1})]$$'
s3.yaxis.axis_label = r'$$ \alpha_{II,I}\ [(s^{-1})]$$'

color_bar = ColorBar(color_mapper=color_mapper, ticker=BasicTicker(),
                     label_standoff=12, border_line_color=None, location=(0,0), title="$$J_{tot}$$ Fold-Change")

s3.add_layout(color_bar, 'right')

idx_X = np.argmin(np.abs(amplitudes - amplitude))
idx_Y = np.argmin(np.abs(frequencies - frequency))
intersect = s3.circle([amplitudes[idx_X]], [frequencies[idx_Y]], size=8, color='red')

 
def update(gamma=10, dG_1=2, dG_2=2, amplitude=2, alpha_41=25, alpha_14=25):
    #s1 update
    J_tot_list = find_J_tot_list(epsilon_list, dG_1, dG_2, alpha_21, alpha_32, gamma)
    idx_eps = np.argmin(np.abs(J_tot_list - np.max(J_tot_list)))
    oscMidpoint = epsilon_list[idx_eps] / 2

    epsilon_1 = float(oscMidpoint + amplitude/2)
    epsilon_2 = float(oscMidpoint - amplitude/2)
    
    # alpha_41 = alpha_14
    p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    p1 = float(p1)
    p2 = float(p2)
    p3 = float(p3)
    p4 = float(p4)
    alpha_12 = alpha_21 * np.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * np.exp((dG_2 - epsilon_1)/ k_bT)


    alpha_45 = alpha_54 * np.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * np.exp((dG_2 - epsilon_2) / k_bT)

    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41


    # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
    # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
    J_12 = np.max([p1 * alpha_12, 0.01])
    J_21 = np.max([p2 * alpha_21, 0.01])

    J_23 = np.max([p2 * alpha_23, 0.01])
    J_32 = np.max([p3 * alpha_32, 0.01])

    J_45 = np.max([p4 * alpha_45, 0.01])
    J_54 = np.max([p5 * alpha_54, 0.01])

    J_56 = np.max([p5 * alpha_56, 0.01])
    J_65 = np.max([p6 * alpha_65, 0.01])

    J_41 = np.max([p4 * alpha_41, 0.01])
    J_14 = np.max([p1 * alpha_14, 0.01])

    J_52 = np.max([p5 * alpha_52, 0.01])
    J_25 = np.max([p2 * alpha_25, 0.01])

    J_36 = np.max([p3 * alpha_36, 0.01])
    J_63 = np.max([p6 * alpha_63, 0.01])

    J_31 = np.max([p3 * gamma, 0.01])
    J_64 = np.max([p6 * gamma, 0.01])




    graph.edge_renderer.data_source.data["line_width"] = np.array([J_12, J_14, J_23, J_25, J_36, J_45, J_56, J_21, J_41, J_32, J_52, J_63, J_54, J_65, J_31, J_64])
    # graph.edge_renderer.data_source.data["line_width"] = [1] * len(s)
    graph.edge_renderer.glyph.line_width = {'field': 'line_width'}
    # S2 update 
    source_1.data = dict(x=probabilities, top=[p1, p2, p3, p4, p5, p6], color=Spectral6)

    # S3 update
    J_tot_list = find_J_tot_list(epsilon_list, dG_1, dG_2, alpha_21, alpha_32, gamma)
    turnover_fold_increase = heatmap(F1, F2, J_tot_list, alpha_21, alpha_32, gamma, dG_1, dG_2, oscMidpoint, amplitude)
    bound = np.min([np.abs(np.max(np.log10(turnover_fold_increase))), np.abs(np.min(np.log10(turnover_fold_increase)))])
    color_mapper.low = 10**-bound
    color_mapper.high = 10**bound
    img.data_source.data['image'] = [turnover_fold_increase]
    idx_X = np.argmin(np.abs(frequencies_1 - alpha_14))
    idx_Y = np.argmin(np.abs(frequencies_2 - alpha_41))
    intersect.data_source.data['y'] = [frequencies_2[idx_Y]]
    intersect.data_source.data['x'] = [frequencies_1[idx_X]]
s1.output_backend = "svg"
s2.output_backend = "svg"
s3.output_backend = "svg"
# plot = row(s1, s3, s2)

plot = row(s1, s3)
interact(update, gamma=(0, 20, 0.1,), dG_1 = (-5,5,0.1), dG_2 = (-5,5,0.1), amplitude=(0,5,0.1), alpha_41=(1e-11,100,1), alpha_14=(1e-11,100,1))
# export_svg(plot, filename="dynamic_steady_state_single.svg")

display(BokehModel(plot))

interactive_python_plots = True
output_notebook()

interactive(children=(FloatSlider(value=10.0, description='gamma', max=20.0), FloatSlider(value=2.0, descripti…

BokehModel(render_bundle={'docs_json': {'91cff1f7-b947-4e09-ab08-c36abb7a58b2': {'version': '3.6.2', 'title': …

Loading BokehJS ...

# Fig 2A,B redo

In [25]:
def heatmap(A, F, J_turnover_total_list, alpha_21, alpha_32, gamma, dG_1, dG_2, oscMidpoint, alpha_41):
    turnover_fold_increase = np.zeros(F.shape)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):

            epsilon_1 = float(oscMidpoint + A[i,j])
            epsilon_2 = float(oscMidpoint - A[i,j])

            alpha_14 = F[i, j]

            p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
            flux = p3 * gamma + p6 * gamma
            turnover_fold_increase[i, j] = flux / max(J_turnover_total_list)
    return turnover_fold_increase

In [26]:
from bokeh.models import LinearColorMapper, ColorBar, BasicTicker
oscMidpoint = 0
k_bT = 1
dG_1 = 2
dG_2 = 2


alpha_21 = 1
alpha_32 = 1

alpha_54 = 1 
alpha_65 = 1 


## S1 definition
s1 = figure(title="Flux Diagram", x_range=(-1.1,1.1), y_range=(-1.1,1.1), tools="hover", tooltips="index: @name", toolbar_location=None, width=208, height=208)
s1.axis.visible = False
N = 6
node_indices = list(range(N))
graph = GraphRenderer()

graph.node_renderer.data_source.add(np.array(node_indices), 'index')
graph.node_renderer.data_source.add(np.array(node_indices) + 1, 'name')
graph.node_renderer.data_source.add(Spectral6, 'color')
graph.node_renderer.glyph = Circle(radius=0.1, fill_color="color")

# s = [0, 2, 1, 1, 2, 3, 1, 2, 3, 3, 0, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 3, 0, 0, 1, 1, 2, 3, 4, 4, 5, 5, 0, 4, 2, 3, 4]
# s = [0, 2, 1, 1, 1, 2, 3, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 0, 1, 1, 4, 4, 5, 5, 0, 4, 2, 3, 4]

s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]

graph.edge_renderer.data_source.data = dict(
    start=s,
    end=e)

### start of layout code
x = [-0.5, 0, 0.5, -0.5, 0, 0.5]
y = [0.5, 0.5, 0.5, -0.5, -0.5, -0.5]
graph_layout = dict(zip(range(len(node_indices)), zip(x, y)))
graph.layout_provider = StaticLayoutProvider(graph_layout=graph_layout)


### Draw quadratic bezier paths
def bezier(start, end, control, steps):
    return [(1-s)**2*start + 2*(1-s)*s*control + s**2*end for s in steps]
vSize = 6
xs, ys = [], []
steps = [i/100. for i in range(100)]
for edge_index in range(len(s)):
    sx, sy = graph_layout[s[edge_index]]
    ex, ey = graph_layout[e[edge_index]]
    match edge_index:
        # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
        # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
        case 0:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
    
        case 1: 
            x = bezier(sx, ex, -0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 2:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.75, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 3:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 4:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 5:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 6:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 7:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 8:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 9: 
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.25, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 10:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 11:
            x = bezier(sx, ex, 0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 12:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.75, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 13:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 14:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, 1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 15:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, -1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        
        case _:
            xs.append(np.linspace(sx, ex, 100))
            ys.append(np.linspace(sy, ey, 100))
    

  
    
graph.edge_renderer.data_source.data['xs'] = xs
graph.edge_renderer.data_source.data['ys'] = ys



s1.renderers.append(graph)

## S2 definition
probabilities = [r'1', r'2', r'3', r'4', r'5', r'6']
s2 = figure(title="State probabilities", x_range=probabilities, y_range=(0,1.0), tools="", toolbar_location=None)
source_1 = bokeh.models.ColumnDataSource(data=dict(x=probabilities, top=[0,0,0,0, 0, 0], color=Spectral6))

s2.vbar(x='x', top='top', width=0.9, source=source_1, color='color')

## S3 definition
max_frequency = 100

max_amplitude = 5
s3 = figure(title="Flux Heatmap", x_range=(0,max_amplitude), y_range=(0, max_frequency), tools="",  toolbar_location=None, width=278, height=208)
n_points = 20

alpha_41 = 250

amplitudes = np.linspace(0, max_amplitude, n_points)
frequencies = np.linspace(1e-5, max_frequency, n_points)
epsilon_list = np.linspace(-5,5,50)

amplitude = 2
frequency = 250

A, F = np.meshgrid(amplitudes, frequencies)
J_tot_list = find_J_tot_list(epsilon_list, dG_1, dG_2, alpha_21, alpha_32, gamma)
turnover_fold_increase = heatmap(A, F, J_tot_list, alpha_21, alpha_32, gamma, dG_1, dG_2, oscMidpoint, alpha_41)
bound = np.max([np.abs(np.max(np.log10(turnover_fold_increase))), np.abs(np.min(np.log10(turnover_fold_increase)))])
color_mapper = LogColorMapper(palette=palette, low=10**-bound, high=10**bound)

img = s3.image(image=[turnover_fold_increase], x=0, y=0, dw=max_amplitude, dh=max_frequency, color_mapper=color_mapper,)
s3.xaxis.axis_label = r'$$ |\epsilon|\ [k_B T]$$'
s3.yaxis.axis_label = r'$$ \alpha_{I,II}\ [(s^{-1})]$$'

color_bar = ColorBar(color_mapper=color_mapper, ticker=BasicTicker(),
                     label_standoff=12, border_line_color=None, location=(0,0), title="$$J_{tot}$$ Fold-Change")

s3.add_layout(color_bar, 'right')

idx_X = np.argmin(np.abs(amplitudes - amplitude))
idx_Y = np.argmin(np.abs(frequencies - frequency))
intersect = s3.circle([amplitudes[idx_X]], [frequencies[idx_Y]], size=8, color='red')

 
def update(gamma=10, dG_1=2, dG_2=2, amplitude=2, alpha_41=25, alpha_14=25):
    #s1 update
    J_tot_list = find_J_tot_list(epsilon_list, dG_1, dG_2, alpha_21, alpha_32, gamma)
    idx_eps = np.argmin(np.abs(J_tot_list - np.max(J_tot_list)))
    oscMidpoint = epsilon_list[idx_eps] / 2

    epsilon_1 = float(oscMidpoint + amplitude/2)
    epsilon_2 = float(oscMidpoint - amplitude/2)
    
    # alpha_41 = alpha_14
    p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    p1 = float(p1)
    p2 = float(p2)
    p3 = float(p3)
    p4 = float(p4)
    alpha_12 = alpha_21 * np.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * np.exp((dG_2 - epsilon_1)/ k_bT)


    alpha_45 = alpha_54 * np.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * np.exp((dG_2 - epsilon_2) / k_bT)

    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41


    # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
    # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
    J_12 = np.max([p1 * alpha_12, 0.01])
    J_21 = np.max([p2 * alpha_21, 0.01])

    J_23 = np.max([p2 * alpha_23, 0.01])
    J_32 = np.max([p3 * alpha_32, 0.01])

    J_45 = np.max([p4 * alpha_45, 0.01])
    J_54 = np.max([p5 * alpha_54, 0.01])

    J_56 = np.max([p5 * alpha_56, 0.01])
    J_65 = np.max([p6 * alpha_65, 0.01])

    J_41 = np.max([p4 * alpha_41, 0.01])
    J_14 = np.max([p1 * alpha_14, 0.01])

    J_52 = np.max([p5 * alpha_52, 0.01])
    J_25 = np.max([p2 * alpha_25, 0.01])

    J_36 = np.max([p3 * alpha_36, 0.01])
    J_63 = np.max([p6 * alpha_63, 0.01])

    J_31 = np.max([p3 * gamma, 0.01])
    J_64 = np.max([p6 * gamma, 0.01])




    graph.edge_renderer.data_source.data["line_width"] = np.array([J_12, J_14, J_23, J_25, J_36, J_45, J_56, J_21, J_41, J_32, J_52, J_63, J_54, J_65, J_31, J_64])
    # graph.edge_renderer.data_source.data["line_width"] = [1] * len(s)
    graph.edge_renderer.glyph.line_width = {'field': 'line_width'}
    # S2 update 
    source_1.data = dict(x=probabilities, top=[p1, p2, p3, p4, p5, p6], color=Spectral6)

    # S3 update
    
    turnover_fold_increase = heatmap(A, F, J_tot_list, alpha_21, alpha_32, gamma, dG_1, dG_2, oscMidpoint, alpha_41)
    bound = np.min([np.abs(np.max(np.log10(turnover_fold_increase))), np.abs(np.min(np.log10(turnover_fold_increase)))])
    color_mapper.low = 10**-bound
    color_mapper.high = 10**bound
    img.data_source.data['image'] = [turnover_fold_increase]
    idx_X = np.argmin(np.abs(amplitudes - amplitude))
    idx_Y = np.argmin(np.abs(frequencies - alpha_14))
    intersect.data_source.data['y'] = [frequencies[idx_Y]]
    intersect.data_source.data['x'] = [amplitudes[idx_X]]
s1.output_backend = "svg"
s2.output_backend = "svg"
s3.output_backend = "svg"
# plot = row(s1, s3, s2)
plot = row(s1, s3,)
interact(update, gamma=(0, 20, 0.1,), dG_1 = (-5,5,0.1), dG_2 = (-5,5,0.1), amplitude=(0,5,0.1), alpha_41=(1e-11,100,1), alpha_14=(1e-11,100,1))

display(BokehModel(plot))

# export_svg(plot, filename="dynamic_steady_state_amp_single.svg")

interactive_python_plots = True
output_notebook()

interactive(children=(FloatSlider(value=10.0, description='gamma', max=20.0), FloatSlider(value=2.0, descripti…

BokehModel(render_bundle={'docs_json': {'3392b0b1-e540-407a-beff-53fd583328f4': {'version': '3.6.2', 'title': …

Loading BokehJS ...

# Fig. 2C

In [27]:
from itertools import product
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

def spanTrees(trs, Edg, all_span_trees, k):
    if k == 0:
        all_span_trees.extend(product(*trs))#Expand parallels edges using the Cartesian product
    
    for i in range(k):
        if Edg[k][i] == []: continue
        trs.append(Edg[k][i])
        Edg[i] = [Edg[i][j] + Edg[k][j] for j in range(i)]#Contraction 
        spanTrees(trs, Edg, all_span_trees, k-1)
        trs.pop()
        [Edg[i][j].pop() for j in range(i) for _ in range(len(Edg[k][j]))]#Deletion

#Helper function
def Spanning_Trees_Generator(g, hch = False):
    if not nx.is_connected(g):
        print("Only for connected graphs")
        return 0
    if nx.is_directed(g): nx.to_undirected(g)
    n = g.number_of_nodes()
    edgs = list(g.edges)
    Edg = [[[] for _ in range(n)] for _ in range(n)]
    mx = len(edgs)
    edgNum = dict() #dictionary designed to store labeled edges where the keys are integer labels for the edges, and the values are tuples representing the ordinary edge definitions (in, out)

    for ed in edgs:
        i, j = sorted(ed)
        Edg[j][i] = [mx]
        edgNum[mx] = ed
        mx -= 1
    all_span_trees = []
    spanTrees([], Edg, all_span_trees, n-1)
  
    if hch:  
        return all_span_trees #Generate only the labeled edges (Keys) 
    else:
        return [nx.Graph(edgNum[k] for k in element) for element in all_span_trees] #Generate spanning trees as NetworkX graphics objects
    
def root_tree_weights(spanning_tree, weights, nodes=None):
    #assumes at least one valid root in the spanning tree implementing one-way fluxes. 
    if nodes is None:
        nodes = set([n for n in spanning_tree])
    
    """ 
    Key insight: if the tree contains a one-way edge, then nothing "upstream" of the one-way edge can be a root.
    Also, all flows "upstream" of one-way edge must flow through the one-way edge.
    
    Else, there would exist at least one edge in spanning_tree that would connect "head" and "tail" subgraphs,
    suggesting a cycle. This is a contradiction.

    Therefore, we can recursively treat tree weight calculation by:
        1. removing one-way edges, if possible
        2. calculating "tail" tree weights, assuming root at one-way edge "tail".
        3. tree weight rooted at node i in "head" = "tail" tree weight * one-way edge weight * "head" tree weight .
        4. Sum all tree weights over all i in the "head".

    A better way to implement this would be to have a set of forbidden edges, then 
    (a). check if a given forbidden edge is in the tree, for all forbidden edges then
    (b). remove all forbidden edges from the tree, then
    (c). find all connected components for each recursive root "tails" of 1-way arrows, then
    (d). account for those sub-trees in the prefactor of the partition sub-function.
    """
    # print('called')
    spanning_tree = spanning_tree.copy()
    prefactor = 1
    if ((3,5) in spanning_tree.edges or (5,3) in spanning_tree.edges) and ((0,2) in spanning_tree.edges or (2,0) in spanning_tree.edges):
        # print('both')
        spanning_tree.remove_edges_from([(3,5), (5,3), (0,2), (2,0)])
        nodes = nodes - nx.node_connected_component(spanning_tree, 5) - nx.node_connected_component(spanning_tree, 2)
        # print(nodes)
        prefactor = prefactor * root_tree_weights(spanning_tree, weights, nodes=set([5])) * weights.get((5, 3)) * root_tree_weights(spanning_tree, weights, nodes=set([2])) * weights.get((2, 0))
    elif (3,5) in spanning_tree.edges or (5,3) in spanning_tree.edges:
        # print('35')
        spanning_tree.remove_edges_from([(3,5), (5,3)])
        nodes = nodes - nx.node_connected_component(spanning_tree, 5)
        prefactor = prefactor * root_tree_weights(spanning_tree, weights, nodes=set([5])) * weights.get((5, 3))
    elif (0,2) in spanning_tree.edges or (2,0) in spanning_tree.edges:
        # print('02')
        spanning_tree.remove_edges_from([(0,2), (2,0)])
        nodes = nodes - nx.node_connected_component(spanning_tree, 2)
        prefactor = prefactor * root_tree_weights(spanning_tree, weights, nodes=set([2])) * weights.get((2, 0))
    # print(prefactor)
    total_weight = 0
    # print(nodes)
    for root in sorted(nodes):
        # Perform BFS to get the tree structure rooted at 'root'
        BFS_list = []
        visited = set()
        queue = [root]
        while queue:
            current = queue.pop(0)
            visited.add(current)
            for neighbor in spanning_tree.neighbors(current):
                if neighbor not in visited and (current, neighbor) in spanning_tree.edges:
                    BFS_list.append((neighbor, current))  # Store as (child, parent)
                    queue.append(neighbor)
        # Calculate the weight of the rooted tree
        # print(BFS_list)
        tree_weight = np.prod([weights.get((u, v)) for u, v in BFS_list]) * prefactor
        # Store or process the tree weight as needed
        # print(tree_weight)
        total_weight += tree_weight
    # print(BFS_list)
    # print(np.prod([weights.get((u, v)) for u, v in BFS_list]))

    return total_weight   

def map_graph_weights(alpha_32=1, alpha_21=1, alpha_65=1, alpha_54=1, alpha_14=1, alpha_41=1, dG_1=2, dG_2=2, gamma=10, epsilon_1=0, epsilon_2=0, k_bT=1):

    alpha_32, alpha_21, alpha_65, alpha_54, alpha_14, alpha_41 = mpmath.mpf(alpha_32), mpmath.mpf(alpha_21), mpmath.mpf(alpha_65), mpmath.mpf(alpha_54), mpmath.mpf(alpha_14), mpmath.mpf(alpha_41)
    dG_1, dG_2, gamma, epsilon_1, epsilon_2, k_bT = mpmath.mpf(dG_1), mpmath.mpf(dG_2), mpmath.mpf(gamma), mpmath.mpf(epsilon_1), mpmath.mpf(epsilon_2), mpmath.mpf(k_bT)
    alpha_12 = alpha_21 * mpmath.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * mpmath.exp((dG_2 - epsilon_1)/ k_bT)

    alpha_54 = alpha_21
    alpha_65 = alpha_32
    alpha_45 = alpha_54 * mpmath.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * mpmath.exp((dG_2 - epsilon_2) / k_bT)



    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41
    d = {        
        (0, 1) : alpha_12,
        (1, 0) : alpha_21,
        (1, 2) : alpha_23,
        (2, 1) : alpha_32,
        (3, 4) : alpha_45,
        (4, 3) : alpha_54,
        (4, 5) : alpha_56,
        (5, 4) : alpha_65,
        (0, 3) : alpha_14,
        (3, 0) : alpha_41,
        (1, 4) : alpha_25,
        (4, 1) : alpha_52,
        (2, 5) : alpha_36,
        (5, 2) : alpha_63,
        (2, 0) : gamma,
        (5, 3) : gamma,
    }

    return d

def dynamic_steady_state_diagram(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma_1, gamma_2, alpha_14, alpha_41, dG_1=2, dG_2=2, k_bT=1):
    '''
    Calculates the turnover rate for a 2-pathway, 6-state catalytic cycle with two
    driven steps (at epsilon_1 and epsilon_2), each with its own driving force
    (dG). Each driven step has its own "completion" rate (gamma_1 and gamma_2).
    All other rate constants are set to 1, except for the reverses of the driven steps,
    which are determined by detailed balance. Transitions between the two pathways are 
    given by (alpha_14, alpha_41), which represent the relative timescales of breaking
    steady-state.
    '''

    epsilon_1, epsilon_2, alpha_21, alpha_32, gamma_1, gamma_2, alpha_14, alpha_41, dG_1, dG_2, k_bT = mpmath.mpf(epsilon_1), mpmath.mpf(epsilon_2), mpmath.mpf(alpha_21), mpmath.mpf(alpha_32), mpmath.mpf(gamma_1), mpmath.mpf(gamma_2), mpmath.mpf(alpha_14), mpmath.mpf(alpha_41), mpmath.mpf(dG_1), mpmath.mpf(dG_2), mpmath.mpf(k_bT)
    
    alpha_54 = alpha_21
    alpha_65 = alpha_32

    alpha_23 = alpha_32 * mpmath.exp((dG_2 - epsilon_1) / k_bT)
    alpha_12 = alpha_21 * mpmath.exp((dG_1 + epsilon_1) / k_bT)

    alpha_56 = alpha_65 * mpmath.exp((dG_2 - epsilon_2) / k_bT)
    alpha_45 = alpha_54 * mpmath.exp((dG_1 + epsilon_2) / k_bT)



    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41

    sigma_1 = alpha_63 * alpha_32 + alpha_36 * alpha_65 + alpha_36 * gamma_2 + \
        alpha_63 * gamma_1 + gamma_1 * gamma_2 + alpha_65 * alpha_32 + \
        alpha_32 * gamma_2 + alpha_65 * gamma_1
    sigma_2 = alpha_41 * alpha_12 + alpha_14 * alpha_45 + alpha_12 * alpha_45
    sigma_3 = 1
    sigma_4 = alpha_14 * alpha_25 * alpha_36 + alpha_21 * alpha_14 * alpha_36 + \
        alpha_14 * alpha_23 * alpha_36 + alpha_12 * alpha_23 * alpha_36 + \
        alpha_32 * alpha_21 * alpha_14 + alpha_12 * alpha_32 * alpha_25 + \
        gamma_1 * alpha_21 * alpha_14 + gamma_1 * alpha_12 * alpha_25 + \
        gamma_1 * alpha_14 * alpha_25 + alpha_23 * gamma_1 * alpha_14 + \
        alpha_12 * alpha_25 * alpha_36 + alpha_32 * alpha_25 * alpha_14
    sigma_5 = alpha_41 * alpha_52 * alpha_63 + alpha_54 * alpha_41 * alpha_63 + \
        alpha_41 * alpha_56 * alpha_63 + alpha_45 * alpha_56 * alpha_63 \
        + alpha_45 * alpha_65 * alpha_52 + alpha_65 * alpha_54 * alpha_41 \
        + gamma_2 * alpha_41 * alpha_54 + gamma_2 * alpha_45 * alpha_52 + \
        gamma_2 * alpha_41 * alpha_52 + alpha_56 * gamma_2 * alpha_41 + \
        alpha_45 * alpha_52 * alpha_63 + alpha_65 * alpha_52 * alpha_41
    sigma_6 = alpha_25 + alpha_23 + alpha_21
    sigma_7 = alpha_52 + alpha_56 + alpha_54
    sigma_8 = alpha_36 + alpha_32 + gamma_1
    sigma_9 = alpha_63 + alpha_65 + gamma_2
    sigma_10 = alpha_45 + alpha_41
    sigma_11 = alpha_14 + alpha_12

    # 3 futile cycles (no conversion)

    Pi_1_plus = alpha_14 * alpha_45 * alpha_52 * alpha_21
    Pi_1_minus = alpha_41 * alpha_54 * alpha_25 * alpha_12

    Pi_2_plus = alpha_25 * alpha_56 * alpha_63 * alpha_32
    Pi_2_minus = alpha_52 * alpha_65 * alpha_36 * alpha_23

    Pi_3_plus = alpha_14 * alpha_45 * alpha_56 * alpha_63 * alpha_32 * alpha_21
    Pi_3_minus = alpha_41 * alpha_54 * alpha_65 * alpha_36 * alpha_23 * alpha_12

    # completion at eps2, eps1

    Pi_4_minus = alpha_45 * alpha_56 * gamma_2

    Pi_5_plus = alpha_12 * alpha_23 * gamma_1

    # completion at eps1, eps2, breaking steady-state, but no net benefit

    Pi_6_plus = alpha_14 * alpha_45 * alpha_56 * alpha_63 * gamma_1

    Pi_7_minus = alpha_41 * alpha_12 * alpha_23 * alpha_36 * gamma_2

    # exactly what you don't want by breaking steady-state

    Pi_8_minus = alpha_41 * alpha_12 * alpha_25 * alpha_56 * gamma_2

    # exactly what you want by breaking steady-state

    Pi_9_plus = alpha_14 * alpha_45 * alpha_52 * alpha_23 * gamma_1

    # exactly what you don't want by breaking steady-state

    Pi_10_plus = alpha_12 * alpha_25 * alpha_56 * alpha_63 * gamma_1

    # exactly what you want by breaking steady-state

    Pi_11_minus = alpha_45 * alpha_52 * alpha_23 * alpha_36 * gamma_2

    J_1 = sigma_1 * (Pi_1_plus - Pi_1_minus)
    J_2 = sigma_2 * (Pi_2_plus - Pi_2_minus)
    J_3 = sigma_3 * (Pi_3_plus - Pi_3_minus) 
    J_4 = sigma_4 * ( - Pi_4_minus) 
    J_5 = sigma_5 * (Pi_5_plus) 
    J_6 = sigma_6 * (Pi_6_plus)
    J_7 = sigma_7 * ( - Pi_7_minus)
    J_8 = sigma_8 * ( - Pi_8_minus)
    J_9 = sigma_9 * (Pi_9_plus) 
    J_10 = sigma_10 * (Pi_10_plus) 
    J_11 = sigma_11 * ( - Pi_11_minus)

    # Calculation of partition function
    G = nx.Graph()
    G.add_nodes_from([2, 1, 0, 3, 4, 5])
    G.add_edges_from([(0, 1), (1, 2), (0,2), (3,4), (4,5), (3,5), (0,3), (1,4), (2,5)])
    spanTreesGraph = Spanning_Trees_Generator(G)

    spanTreesGraph[14] = np.nan
    spanTreesGraph[17] = np.nan
    spanTreesGraph[27] = np.nan
    spanTreesGraph[30] = np.nan
    spanTreesGraph[33] = np.nan
    spanTreesGraph[58] = np.nan
    spanTreesGraph[61] = np.nan
    spanTreesGraph[70] = np.nan
    spanTreesGraph[73] = np.nan
    spanTreesGraph = [g for g in spanTreesGraph if type(g) != float]

    weights = map_graph_weights(alpha_14=alpha_14, alpha_41=alpha_41, epsilon_1=epsilon_1, epsilon_2=epsilon_2, gamma=gamma_1, dG_1=dG_1, dG_2=dG_2)
    tree_weights = [root_tree_weights(spanTreesGraph[i], weights, nodes=None) for i in range(len(spanTreesGraph))]
    sigma = np.sum(tree_weights)   
        
    J_1 = J_1 / sigma
    J_2 = J_2 / sigma
    J_3 = J_3 / sigma
    J_4 = J_4 / sigma
    J_5 = J_5 / sigma
    J_6 = J_6 / sigma
    J_7 = J_7 / sigma

    J_8 = J_8 / sigma
    J_9 = J_9 / sigma
    J_10 = J_10 / sigma
    J_11 = J_11 / sigma

    return J_1, J_2, J_3, J_4, J_5, J_6, J_7, J_8, J_9, J_10, J_11

def solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma_1, gamma_2, alpha_14, alpha_41, dG_1=2, dG_2=2, k_bT=1):
    '''
    Solves for the steady-state probabilities of each state in the 6-state system.
    '''
    epsilon_1, epsilon_2, alpha_21, alpha_32, gamma_1, gamma_2, alpha_14, alpha_41, dG_1, dG_2, k_bT = mpmath.mpf(epsilon_1), mpmath.mpf(epsilon_2), \
          mpmath.mpf(alpha_21), mpmath.mpf(alpha_32),\
              mpmath.mpf(gamma_1), mpmath.mpf(gamma_2), \
                mpmath.mpf(alpha_14), mpmath.mpf(alpha_41), \
                    mpmath.mpf(dG_1), mpmath.mpf(dG_2), \
                        mpmath.mpf(k_bT)
    
    alpha_12 = alpha_21 * mpmath.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * mpmath.exp((dG_2 - epsilon_1)/ k_bT)

    alpha_54 = alpha_21
    alpha_65 = alpha_32
    alpha_45 = alpha_54 * mpmath.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * mpmath.exp((dG_2 - epsilon_2) / k_bT)



    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41

    a = np.array([[-(alpha_12 + alpha_14), alpha_21, gamma_1, alpha_41, 0 , 0],
                [alpha_12, -(alpha_21 + alpha_23 + alpha_25), alpha_32, 0, alpha_52, 0],
                [0, alpha_23, -(gamma_1 + alpha_32 + alpha_36), 0, 0, alpha_63],
                [alpha_14, 0, 0, -(alpha_41 + alpha_45), alpha_54, gamma_2],
                [0, alpha_25, 0, alpha_45, -(alpha_52 + alpha_54 + alpha_56), alpha_65],
                [0, 0, alpha_36, 0, alpha_56, -(gamma_2 + alpha_63 + alpha_65)]], dtype=float)  
    a[-1:,] = mpmath.mpf(1)
    bnp = np.zeros(a.shape[0])
    bnp[-1] = mpmath.mpf(1)

    a = mpmath.matrix(a.tolist())
    b = bnp.tolist()
    sol = np.array(mpmath.lu_solve(a,b)).astype(float)
    return sol

In [28]:
from bokeh.models import LinearColorMapper, ColorBar, BasicTicker
from bokeh.palettes import Spectral6
oscMidpoint = 0
k_bT = 1
dG_1 = 2
dG_2 = 2


alpha_21 = 1
alpha_32 = 1

alpha_54 = 1 
alpha_65 = 1 


## S1 definition
s1 = figure(title="Flux Diagram", x_range=(-1.1,1.1), y_range=(-1.1,1.1), tools="hover", tooltips="index: @name", toolbar_location=None, width=208, height=208)
s1.axis.visible = False
N = 6
node_indices = list(range(N))
graph = GraphRenderer()

graph.node_renderer.data_source.add(np.array(node_indices), 'index')
graph.node_renderer.data_source.add(np.array(node_indices) + 1, 'name')
graph.node_renderer.data_source.add(Spectral6, 'color')
graph.node_renderer.glyph = Circle(radius=0.1, fill_color="color")

# s = [0, 2, 1, 1, 2, 3, 1, 2, 3, 3, 0, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 3, 0, 0, 1, 1, 2, 3, 4, 4, 5, 5, 0, 4, 2, 3, 4]
# s = [0, 2, 1, 1, 1, 2, 3, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 0, 1, 1, 4, 4, 5, 5, 0, 4, 2, 3, 4]

s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]

graph.edge_renderer.data_source.data = dict(
    start=s,
    end=e)

### start of layout code
x = [-0.5, 0, 0.5, -0.5, 0, 0.5]
y = [0.5, 0.5, 0.5, -0.5, -0.5, -0.5]
graph_layout = dict(zip(range(len(node_indices)), zip(x, y)))
graph.layout_provider = StaticLayoutProvider(graph_layout=graph_layout)


### Draw quadratic bezier paths
def bezier(start, end, control, steps):
    return [(1-s)**2*start + 2*(1-s)*s*control + s**2*end for s in steps]
vSize = 6
xs, ys = [], []
steps = [i/100. for i in range(100)]
for edge_index in range(len(s)):
    sx, sy = graph_layout[s[edge_index]]
    ex, ey = graph_layout[e[edge_index]]
    match edge_index:
        # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
        # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
        case 0:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
    
        case 1: 
            x = bezier(sx, ex, -0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 2:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.75, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 3:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 4:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 5:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 6:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 7:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 8:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 9: 
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.25, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 10:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 11:
            x = bezier(sx, ex, 0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 12:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.75, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 13:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 14:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, 1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 15:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, -1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        
        case _:
            xs.append(np.linspace(sx, ex, 100))
            ys.append(np.linspace(sy, ey, 100))
    

  
    
graph.edge_renderer.data_source.data['xs'] = xs
graph.edge_renderer.data_source.data['ys'] = ys



s1.renderers.append(graph)

## S2 definition
probabilities = [r'1', r'2', r'3', r'4', r'5', r'6']
s2 = figure(title="State probabilities", x_range=probabilities, y_range=(0,1.0), tools="", toolbar_location=None, width=208, height=208)
source_1 = bokeh.models.ColumnDataSource(data=dict(x=probabilities, top=[0,0,0,0, 0, 0], color=Spectral6))

s2.vbar(x='x', top='top', width=0.9, source=source_1, color='color')

## S3 definition
cats = [ r'steady-state', r'dynamic steady-state', r'futile cycles']
fluxes = [r'nonenhanced', r'enhanced',  r'impeded', r'futile', r'optimum']
s3 = figure(title="Cycle fluxes", x_range=cats, y_range=(0,5), toolbar_location=None, tools="hover", tooltips="$name: @$name", width=288, height=208)
data = {'cats': cats,
        'optimum': [None,0, None],
        'nonenhanced': [0,0, None],
        'enhanced': [0,0, None],
        'impeded': [0,0, None],
        'futile': [None, None, 0],
        }
source_2 = bokeh.models.ColumnDataSource(data=data)
s3.vbar_stack(fluxes, x='cats', width=0.9, source=source_2, color=Spectral6[:-1], line_color=None)
# s3.vbar_stack(fluxes, x='cats', width=0.9, source=source_2, color=Spectral6[:-1], line_color=None, legend_label=fluxes)
# s3.legend.location = "top_left"
# s3.legend.orientation = "horizontal"
s3.yaxis.axis_label = "Flux [$$s^{-1}$$]"
s3.xaxis.major_label_text_font_size = '6pt'
# s3.legend.label_text_font_size = '6pt'
# s3.legend.items[3].visible = False
# s3.legend.items[4].visible = False


 
def update(gamma=10, dG_1=2, dG_2=2, amplitude=2, alpha_41=25, alpha_14=25):
    #s1 update
    J_tot_list = find_J_tot_list(epsilon_list, dG_1, dG_2, alpha_21, alpha_32, gamma)
    idx_eps = np.argmin(np.abs(J_tot_list - np.max(J_tot_list)))
    oscMidpoint = epsilon_list[idx_eps] / 2

    epsilon_1 = float(oscMidpoint + amplitude/2)
    epsilon_2 = float(oscMidpoint - amplitude/2)
    
    # alpha_41 = alpha_14
    p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    p1 = float(p1)
    p2 = float(p2)
    p3 = float(p3)
    p4 = float(p4)
    alpha_12 = alpha_21 * np.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * np.exp((dG_2 - epsilon_1)/ k_bT)


    alpha_45 = alpha_54 * np.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * np.exp((dG_2 - epsilon_2) / k_bT)

    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41


    # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
    # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
    J_12 = np.max([p1 * alpha_12, 0.01])
    J_21 = np.max([p2 * alpha_21, 0.01])

    J_23 = np.max([p2 * alpha_23, 0.01])
    J_32 = np.max([p3 * alpha_32, 0.01])

    J_45 = np.max([p4 * alpha_45, 0.01])
    J_54 = np.max([p5 * alpha_54, 0.01])

    J_56 = np.max([p5 * alpha_56, 0.01])
    J_65 = np.max([p6 * alpha_65, 0.01])

    J_41 = np.max([p4 * alpha_41, 0.01])
    J_14 = np.max([p1 * alpha_14, 0.01])

    J_52 = np.max([p5 * alpha_52, 0.01])
    J_25 = np.max([p2 * alpha_25, 0.01])

    J_36 = np.max([p3 * alpha_36, 0.01])
    J_63 = np.max([p6 * alpha_63, 0.01])

    J_31 = np.max([p3 * gamma, 0.01])
    J_64 = np.max([p6 * gamma, 0.01])




    graph.edge_renderer.data_source.data["line_width"] = np.array([J_12, J_14, J_23, J_25, J_36, J_45, J_56, J_21, J_41, J_32, J_52, J_63, J_54, J_65, J_31, J_64])
    graph.edge_renderer.data_source.data["line_color"] = np.array(['black', 'black', 'red', 'black', 'red', 'red', 'black', 'black', 'black', 'black', 'red', 'black', 'black', 'black', 'black', 'red'])
    # graph.edge_renderer.data_source.data["line_width"] = [1] * len(s)
    graph.edge_renderer.glyph.line_width = {'field': 'line_width'}
    graph.edge_renderer.glyph.line_color = {'field': 'line_color'}
    # S2 update 
    source_1.data = dict(x=probabilities, top=[p1, p2, p3, p4, p5, p6], color=Spectral6)

    # S3 update
    J_1, J_2, J_3, J_4, J_5, J_6, J_7, J_8, J_9, J_10, J_11 = dynamic_steady_state_diagram(epsilon_1, epsilon_2, alpha_21, alpha_32, \
                                                                    gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    J_1 = float(J_1)
    J_2 = float(J_2)
    J_3 = float(J_3)
    J_4 = float(J_4)
    J_5 = float(J_5)
    J_6 = float(J_6)
    J_7 = float(J_7)
    J_8 = float(J_8)
    J_9 = float(J_9)
    J_10 = float(J_10)
    J_11 = float(J_11)

    J_enhanced = np.abs(J_10) + np.abs(J_8)
    J_nonenhanced = np.abs(J_4) + np.abs(J_5) + np.abs(J_6) + np.abs(J_7)
    J_impeded = np.abs(J_9) + np.abs(J_11)
    J_futile = np.abs(J_1) + np.abs(J_2) + np.abs(J_3)
    J_steady_state_optimum = np.max(J_tot_list)
    
    source_2.data = {'cats': cats,
        'optimum': [J_steady_state_optimum, None, None],
        'enhanced': [None, J_enhanced, None],
        'nonenhanced': [None, J_nonenhanced, None],
        'impeded': [None, J_impeded, None],
        'futile': [None,None, J_futile],
        }

s1.output_backend = "svg"
s2.output_backend = "svg"
s3.output_backend = "svg"
plot = row(s1, s3, )
# plot = row(s1, s2, s3)
interact(update, gamma=(0, 20, 0.1,), dG_1 = (-5,5,0.1), dG_2 = (-5,5,0.1), amplitude=(0,5,0.1), alpha_41=(1e-11,100,1), alpha_14=(1e-11,100,1))

display(BokehModel(plot))

export_svg(plot, filename="dynamic_steady_state_barplot.svg")

interactive_python_plots = True
output_notebook()

interactive(children=(FloatSlider(value=10.0, description='gamma', max=20.0), FloatSlider(value=2.0, descripti…

BokehModel(render_bundle={'docs_json': {'c85e5dc2-b76f-4999-924b-8c5f084e7897': {'version': '3.6.2', 'title': …

RuntimeError: Neither firefox and geckodriver nor a variant of chromium browser and chromedriver are available on system PATH. You can install the former with 'conda install -c conda-forge firefox geckodriver'.

In [ ]:
from bokeh.models import LinearColorMapper, ColorBar, BasicTicker
from bokeh.palettes import Spectral6
oscMidpoint = 0
k_bT = 1
dG_1 = 2
dG_2 = 2


alpha_21 = 1
alpha_32 = 1

alpha_54 = 1 
alpha_65 = 1 


## S1 definition
s1 = figure(title="Flux Diagram", x_range=(-1.1,1.1), y_range=(-1.1,1.1), tools="hover", tooltips="index: @name", toolbar_location=None, width=208, height=208)
s1.axis.visible = False
N = 6
node_indices = list(range(N))
graph = GraphRenderer()

graph.node_renderer.data_source.add(np.array(node_indices), 'index')
graph.node_renderer.data_source.add(np.array(node_indices) + 1, 'name')
graph.node_renderer.data_source.add(Spectral6, 'color')
graph.node_renderer.glyph = Circle(radius=0.1, fill_color="color")

# s = [0, 2, 1, 1, 2, 3, 1, 2, 3, 3, 0, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 3, 0, 0, 1, 1, 2, 3, 4, 4, 5, 5, 0, 4, 2, 3, 4]
# s = [0, 2, 1, 1, 1, 2, 3, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 0, 1, 1, 4, 4, 5, 5, 0, 4, 2, 3, 4]

s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]

graph.edge_renderer.data_source.data = dict(
    start=s,
    end=e)

### start of layout code
x = [-0.5, 0, 0.5, -0.5, 0, 0.5]
y = [0.5, 0.5, 0.5, -0.5, -0.5, -0.5]
graph_layout = dict(zip(range(len(node_indices)), zip(x, y)))
graph.layout_provider = StaticLayoutProvider(graph_layout=graph_layout)


### Draw quadratic bezier paths
def bezier(start, end, control, steps):
    return [(1-s)**2*start + 2*(1-s)*s*control + s**2*end for s in steps]
vSize = 6
xs, ys = [], []
steps = [i/100. for i in range(100)]
for edge_index in range(len(s)):
    sx, sy = graph_layout[s[edge_index]]
    ex, ey = graph_layout[e[edge_index]]
    match edge_index:
        # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
        # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
        case 0:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
    
        case 1: 
            x = bezier(sx, ex, -0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 2:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.75, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 3:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 4:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 5:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 6:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 7:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 8:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 9: 
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.25, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 10:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 11:
            x = bezier(sx, ex, 0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 12:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.75, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 13:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 14:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, 1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 15:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, -1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        
        case _:
            xs.append(np.linspace(sx, ex, 100))
            ys.append(np.linspace(sy, ey, 100))
    

  
    
graph.edge_renderer.data_source.data['xs'] = xs
graph.edge_renderer.data_source.data['ys'] = ys



s1.renderers.append(graph)

## S2 definition
probabilities = [r'1', r'2', r'3', r'4', r'5', r'6']
s2 = figure(title="State probabilities", x_range=probabilities, y_range=(0,1.0), tools="", toolbar_location=None, width=208, height=208)
source_1 = bokeh.models.ColumnDataSource(data=dict(x=probabilities, top=[0,0,0,0, 0, 0], color=Spectral6))

s2.vbar(x='x', top='top', width=0.9, source=source_1, color='color')

## S3 definition
cats = [ r'steady-state', r'dynamic steady-state', r'futile cycles']
fluxes = [r'nonenhanced', r'enhanced',  r'impeded', r'futile', r'optimum']
s3 = figure(title="Cycle fluxes", x_range=cats, y_range=(0,5), toolbar_location=None, tools="hover", tooltips="$name: @$name", width=288, height=208)
data = {'cats': cats,
        'optimum': [None,0, None],
        'nonenhanced': [0,0, None],
        'enhanced': [0,0, None],
        'impeded': [0,0, None],
        'futile': [None, None, 0],
        }
source_2 = bokeh.models.ColumnDataSource(data=data)
s3.vbar_stack(fluxes, x='cats', width=0.9, source=source_2, color=Spectral6[:-1], line_color=None, legend_label=fluxes)
s3.legend.location = "top_left"
s3.legend.orientation = "horizontal"
s3.yaxis.axis_label = "Flux [$$s^{-1}$$]"
s3.xaxis.major_label_text_font_size = '6pt'
s3.legend.label_text_font_size = '6pt'
s3.legend.items[3].visible = False
s3.legend.items[4].visible = False


 
def update(gamma=10, dG_1=2, dG_2=2, amplitude=2, alpha_41=25, alpha_14=25):
    #s1 update
    J_tot_list = find_J_tot_list(epsilon_list, dG_1, dG_2, alpha_21, alpha_32, gamma)
    idx_eps = np.argmin(np.abs(J_tot_list - np.max(J_tot_list)))
    oscMidpoint = epsilon_list[idx_eps] / 2

    epsilon_1 = float(oscMidpoint + amplitude/2)
    epsilon_2 = float(oscMidpoint - amplitude/2)
    
    # alpha_41 = alpha_14
    p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    p1 = float(p1)
    p2 = float(p2)
    p3 = float(p3)
    p4 = float(p4)
    alpha_12 = alpha_21 * np.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * np.exp((dG_2 - epsilon_1)/ k_bT)


    alpha_45 = alpha_54 * np.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * np.exp((dG_2 - epsilon_2) / k_bT)

    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41


    # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
    # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
    J_12 = np.max([p1 * alpha_12, 0.01])
    J_21 = np.max([p2 * alpha_21, 0.01])

    J_23 = np.max([p2 * alpha_23, 0.01])
    J_32 = np.max([p3 * alpha_32, 0.01])

    J_45 = np.max([p4 * alpha_45, 0.01])
    J_54 = np.max([p5 * alpha_54, 0.01])

    J_56 = np.max([p5 * alpha_56, 0.01])
    J_65 = np.max([p6 * alpha_65, 0.01])

    J_41 = np.max([p4 * alpha_41, 0.01])
    J_14 = np.max([p1 * alpha_14, 0.01])

    J_52 = np.max([p5 * alpha_52, 0.01])
    J_25 = np.max([p2 * alpha_25, 0.01])

    J_36 = np.max([p3 * alpha_36, 0.01])
    J_63 = np.max([p6 * alpha_63, 0.01])

    J_31 = np.max([p3 * gamma, 0.01])
    J_64 = np.max([p6 * gamma, 0.01])




    graph.edge_renderer.data_source.data["line_width"] = np.array([J_12, J_14, J_23, J_25, J_36, J_45, J_56, J_21, J_41, J_32, J_52, J_63, J_54, J_65, J_31, J_64])
    # graph.edge_renderer.data_source.data["line_width"] = [1] * len(s)
    graph.edge_renderer.glyph.line_width = {'field': 'line_width'}
    # S2 update 
    source_1.data = dict(x=probabilities, top=[p1, p2, p3, p4, p5, p6], color=Spectral6)

    # S3 update
    J_1, J_2, J_3, J_4, J_5, J_6, J_7, J_8, J_9, J_10, J_11 = dynamic_steady_state_diagram(epsilon_1, epsilon_2, alpha_21, alpha_32, \
                                                                    gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    J_1 = float(J_1)
    J_2 = float(J_2)
    J_3 = float(J_3)
    J_4 = float(J_4)
    J_5 = float(J_5)
    J_6 = float(J_6)
    J_7 = float(J_7)
    J_8 = float(J_8)
    J_9 = float(J_9)
    J_10 = float(J_10)
    J_11 = float(J_11)

    J_enhanced = np.abs(J_10) + np.abs(J_8)
    J_nonenhanced = np.abs(J_4) + np.abs(J_5) + np.abs(J_6) + np.abs(J_7)
    J_impeded = np.abs(J_9) + np.abs(J_11)
    J_futile = np.abs(J_1) + np.abs(J_2) + np.abs(J_3)
    J_steady_state_optimum = np.max(J_tot_list)
    
    source_2.data = {'cats': cats,
        'optimum': [J_steady_state_optimum, None, None],
        'enhanced': [None, J_enhanced, None],
        'nonenhanced': [None, J_nonenhanced, None],
        'impeded': [None, J_impeded, None],
        'futile': [None,None, J_futile],
        }

s1.output_backend = "svg"
s2.output_backend = "svg"
s3.output_backend = "svg"
plot = row(s1, s3, )
# plot = row(s1, s2, s3)
interact(update, gamma=(0, 20, 0.1,), dG_1 = (-5,5,0.1), dG_2 = (-5,5,0.1), amplitude=(0,5,0.1), alpha_41=(1e-11,100,1), alpha_14=(1e-11,100,1))

display(BokehModel(plot))

# export_svg(plot, filename="dynamic_steady_state_barplot.svg")

interactive_python_plots = True
output_notebook()

interactive(children=(FloatSlider(value=10.0, description='gamma', max=20.0), FloatSlider(value=2.0, descripti…

BokehModel(render_bundle={'docs_json': {'af74ce17-8487-4143-a8d0-2e2e4105a2b1': {'version': '3.6.2', 'title': …

Loading BokehJS ...

# Implementing additional futile cycle contributions (pathway i -> ii -> i)

In [29]:
def dynamic_steady_state_diagram(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma_1, gamma_2, alpha_14, alpha_41, dG_1=2, dG_2=2, k_bT=1):
    '''
    Calculates the turnover rate for a 2-pathway, 6-state catalytic cycle with two
    driven steps (at epsilon_1 and epsilon_2), each with its own driving force
    (dG). Each driven step has its own "completion" rate (gamma_1 and gamma_2).
    All other rate constants are set to 1, except for the reverses of the driven steps,
    which are determined by detailed balance. Transitions between the two pathways are 
    given by (alpha_14, alpha_41), which represent the relative timescales of breaking
    steady-state.'''

    epsilon_1, epsilon_2, alpha_21, alpha_32, gamma_1, gamma_2, alpha_14, alpha_41, dG_1, dG_2, k_bT = mpmath.mpf(epsilon_1), mpmath.mpf(epsilon_2), mpmath.mpf(alpha_21), mpmath.mpf(alpha_32), mpmath.mpf(gamma_1), mpmath.mpf(gamma_2), mpmath.mpf(alpha_14), mpmath.mpf(alpha_41), mpmath.mpf(dG_1), mpmath.mpf(dG_2), mpmath.mpf(k_bT)
    
    alpha_54 = alpha_21
    alpha_65 = alpha_32

    alpha_23 = alpha_32 * mpmath.exp((dG_2 - epsilon_1) / k_bT)
    alpha_12 = alpha_21 * mpmath.exp((dG_1 + epsilon_1) / k_bT)

    alpha_56 = alpha_65 * mpmath.exp((dG_2 - epsilon_2) / k_bT)
    alpha_45 = alpha_54 * mpmath.exp((dG_1 + epsilon_2) / k_bT)



    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41

    sigma_1 = alpha_63 * alpha_32 + alpha_36 * alpha_65 + alpha_36 * gamma_2 + \
        alpha_63 * gamma_1 + gamma_1 * gamma_2 + alpha_65 * alpha_32 + \
        alpha_32 * gamma_2 + alpha_65 * gamma_1
    sigma_2 = alpha_41 * alpha_12 + alpha_14 * alpha_45 + alpha_12 * alpha_45
    sigma_3 = 1
    sigma_4 = alpha_14 * alpha_25 * alpha_36 + alpha_21 * alpha_14 * alpha_36 + \
        alpha_14 * alpha_23 * alpha_36 + alpha_12 * alpha_23 * alpha_36 + \
        alpha_32 * alpha_21 * alpha_14 + alpha_12 * alpha_32 * alpha_25 + \
        gamma_1 * alpha_21 * alpha_14 + gamma_1 * alpha_12 * alpha_25 + \
        gamma_1 * alpha_14 * alpha_25 + alpha_23 * gamma_1 * alpha_14 + \
        alpha_12 * alpha_25 * alpha_36 + alpha_32 * alpha_25 * alpha_14
    sigma_5 = alpha_41 * alpha_52 * alpha_63 + alpha_54 * alpha_41 * alpha_63 + \
        alpha_41 * alpha_56 * alpha_63 + alpha_45 * alpha_56 * alpha_63 \
        + alpha_45 * alpha_65 * alpha_52 + alpha_65 * alpha_54 * alpha_41 \
        + gamma_2 * alpha_41 * alpha_54 + gamma_2 * alpha_45 * alpha_52 + \
        gamma_2 * alpha_41 * alpha_52 + alpha_56 * gamma_2 * alpha_41 + \
        alpha_45 * alpha_52 * alpha_63 + alpha_65 * alpha_52 * alpha_41
    sigma_6 = alpha_25 + alpha_23 + alpha_21
    sigma_7 = alpha_52 + alpha_56 + alpha_54
    sigma_8 = alpha_36 + alpha_32 + gamma_1
    sigma_9 = alpha_63 + alpha_65 + gamma_2
    sigma_10 = alpha_45 + alpha_41
    sigma_11 = alpha_14 + alpha_12

    # fluxes from i -> ii -> i
    sigma_f1 = alpha_21 * alpha_54 * alpha_32 * alpha_63 \
        + alpha_21 * alpha_32 * alpha_63 * alpha_56 \
        + alpha_23 * alpha_36 * alpha_65 * alpha_54 \
        + alpha_21 * alpha_36 * alpha_65 * alpha_54 \
        + alpha_32 * alpha_21 * alpha_65 * alpha_54 \
        + alpha_21 * alpha_52 * alpha_32 * alpha_63 \
        + alpha_54 * alpha_25 * alpha_65 * alpha_36 \
        + alpha_21 * alpha_52 * alpha_65 * alpha_36 \
        + alpha_54 * alpha_25 * alpha_32 * alpha_63 \
        + alpha_54 * alpha_25 * alpha_65 * alpha_32 \
        + alpha_21 * alpha_32 * alpha_52 * alpha_65 \
        + alpha_21 * alpha_54 * alpha_36 * gamma_2 \
        + alpha_21 * alpha_54 * gamma_1 * alpha_63 \
        + alpha_21 * alpha_54 * gamma_1 * gamma_2 \
        + alpha_21 * alpha_54 * alpha_65 * gamma_1 \
        + alpha_21 * alpha_54 * alpha_32 * gamma_2 \
        + alpha_21 * alpha_52 * gamma_1 * gamma_2 \
        + alpha_21 * alpha_52 * alpha_65 * gamma_1 \
        + alpha_54 * alpha_25 * gamma_1 * gamma_2 \
        + alpha_54 * alpha_25 * alpha_32 * gamma_2 \
        + alpha_54 * alpha_25 * alpha_65 * gamma_1 \
        + alpha_54 * alpha_25 * alpha_63 * gamma_1 \
        + alpha_54 * alpha_25 * alpha_36 * gamma_2 \
        + alpha_21 * alpha_52 * gamma_1 * alpha_63 \
        + alpha_21 * alpha_52 * gamma_2 * alpha_36 \
        + alpha_21 * alpha_52 * alpha_32 * gamma_2
    
    sigma_f2 = alpha_12 * alpha_32 * alpha_45 * alpha_65 \
    + alpha_41 * alpha_12 * alpha_32 * alpha_65 \
    + alpha_14 * alpha_45 * alpha_32 * alpha_65 \
    + alpha_12 * alpha_45 * alpha_63 * alpha_32 \
    + alpha_12 * alpha_45 * alpha_65 * alpha_36 \
    + alpha_14 * alpha_45 * alpha_65 * alpha_36 \
    + alpha_41 * alpha_12 * alpha_36 * alpha_65 \
    + alpha_41 * alpha_12 * alpha_32 * alpha_63 \
    + alpha_14 * alpha_45 * alpha_32 * alpha_63 \
    + alpha_12 * alpha_45 * gamma_1 * gamma_2 \
    + alpha_14 * alpha_45 * gamma_1 * gamma_2 \
    + alpha_41 * alpha_12 * gamma_1 * gamma_2 \
    + alpha_12 * alpha_45 * alpha_32 * gamma_2 \
    + alpha_14 * alpha_45 * alpha_32 * gamma_2 \
    + alpha_12 * alpha_32 * alpha_41 * gamma_2 \
    + alpha_12 * alpha_45 * alpha_36 * gamma_2 \
    + alpha_14 * alpha_45 * alpha_36 * gamma_2 \
    + alpha_41 * alpha_12 * alpha_36 * gamma_2 \
    + alpha_12 * alpha_45 * alpha_63 * gamma_1 \
    + alpha_12 * alpha_41 * alpha_63 * gamma_1 \
    + alpha_14 * alpha_45 * alpha_63 * gamma_1 \
    + alpha_12 * alpha_45 * alpha_65 * gamma_1 \
    + alpha_41 * alpha_12 * alpha_65 * gamma_1 \
    + alpha_14 * alpha_45 * alpha_65 * gamma_1 

    sigma_f3 = alpha_41 * alpha_12 * alpha_23 * alpha_56 \
    + alpha_54 * alpha_41 * alpha_12 * alpha_23 \
    + alpha_21 * alpha_14 * alpha_45 * alpha_56 \
    + alpha_14 * alpha_45 * alpha_56 * alpha_23 \
    + alpha_12 * alpha_23 * alpha_45 * alpha_56 \
    + alpha_41 * alpha_12 * alpha_23 * alpha_52 \
    + alpha_14 * alpha_45 * alpha_56 * alpha_25 \
    + alpha_14 * alpha_45 * alpha_52 * alpha_23 \
    + alpha_41 * alpha_12 * alpha_25 * alpha_56 \
    + alpha_12 * alpha_25 * alpha_45 * alpha_56 \
    + alpha_12 * alpha_23 * alpha_52 * alpha_45    

    # 3 futile cycles (no conversion)

    Pi_1_plus = alpha_14 * alpha_45 * alpha_52 * alpha_21
    Pi_1_minus = alpha_41 * alpha_54 * alpha_25 * alpha_12

    Pi_2_plus = alpha_25 * alpha_56 * alpha_63 * alpha_32
    Pi_2_minus = alpha_52 * alpha_65 * alpha_36 * alpha_23

    Pi_3_plus = alpha_14 * alpha_45 * alpha_56 * alpha_63 * alpha_32 * alpha_21
    Pi_3_minus = alpha_41 * alpha_54 * alpha_65 * alpha_36 * alpha_23 * alpha_12

    # completion at eps2, eps1

    Pi_4_minus = alpha_45 * alpha_56 * gamma_2

    Pi_5_plus = alpha_12 * alpha_23 * gamma_1

    # completion at eps1, eps2, breaking steady-state, but no net benefit

    Pi_6_plus = alpha_14 * alpha_45 * alpha_56 * alpha_63 * gamma_1

    Pi_7_minus = alpha_41 * alpha_12 * alpha_23 * alpha_36 * gamma_2

    # exactly what you don't want by breaking steady-state

    Pi_8_minus = alpha_41 * alpha_12 * alpha_25 * alpha_56 * gamma_2

    # exactly what you want by breaking steady-state

    Pi_9_plus = alpha_14 * alpha_45 * alpha_52 * alpha_23 * gamma_1

    # exactly what you don't want by breaking steady-state

    Pi_10_plus = alpha_12 * alpha_25 * alpha_56 * alpha_63 * gamma_1

    # exactly what you want by breaking steady-state

    Pi_11_minus = alpha_45 * alpha_52 * alpha_23 * alpha_36 * gamma_2

    # fluxes from i -> ii -> i
    Pi_f1 = alpha_14 * alpha_41
    Pi_f2 = alpha_25 * alpha_52
    Pi_f3 = alpha_36 * alpha_63

    J_1 = sigma_1 * (Pi_1_plus - Pi_1_minus)
    J_2 = sigma_2 * (Pi_2_plus - Pi_2_minus)
    J_3 = sigma_3 * (Pi_3_plus - Pi_3_minus) 
    J_4 = sigma_4 * ( - Pi_4_minus) 
    J_5 = sigma_5 * (Pi_5_plus) 
    J_6 = sigma_6 * (Pi_6_plus)
    J_7 = sigma_7 * ( - Pi_7_minus)
    J_8 = sigma_8 * ( - Pi_8_minus)
    J_9 = sigma_9 * (Pi_9_plus) 
    J_10 = sigma_10 * (Pi_10_plus) 
    J_11 = sigma_11 * ( - Pi_11_minus)

    J_f1 = sigma_f1 * Pi_f1
    J_f2 = sigma_f2 * Pi_f2
    J_f3 = sigma_f3 * Pi_f3

    # Calculation of partition function
    G = nx.Graph()
    G.add_nodes_from([2, 1, 0, 3, 4, 5])
    G.add_edges_from([(0, 1), (1, 2), (0,2), (3,4), (4,5), (3,5), (0,3), (1,4), (2,5)])
    spanTreesGraph = Spanning_Trees_Generator(G)

    spanTreesGraph[14] = np.nan
    spanTreesGraph[17] = np.nan
    spanTreesGraph[27] = np.nan
    spanTreesGraph[30] = np.nan
    spanTreesGraph[33] = np.nan
    spanTreesGraph[58] = np.nan
    spanTreesGraph[61] = np.nan
    spanTreesGraph[70] = np.nan
    spanTreesGraph[73] = np.nan
    spanTreesGraph = [g for g in spanTreesGraph if type(g) != float]

    weights = map_graph_weights(alpha_14=alpha_14, alpha_41=alpha_41, epsilon_1=epsilon_1, epsilon_2=epsilon_2, gamma=gamma_1, dG_1=dG_1, dG_2=dG_2)
    tree_weights = [root_tree_weights(spanTreesGraph[i], weights, nodes=None) for i in range(len(spanTreesGraph))]
    sigma = np.sum(tree_weights)   
        
    J_1 = J_1 / sigma
    J_2 = J_2 / sigma
    J_3 = J_3 / sigma
    J_4 = J_4 / sigma
    J_5 = J_5 / sigma
    J_6 = J_6 / sigma
    J_7 = J_7 / sigma

    J_8 = J_8 / sigma
    J_9 = J_9 / sigma
    J_10 = J_10 / sigma
    J_11 = J_11 / sigma

    J_f1 = J_f1 / sigma
    J_f2 = J_f2 / sigma
    J_f3 = J_f3 / sigma

    return J_1, J_2, J_3, J_4, J_5, J_6, J_7, J_8, J_9, J_10, J_11, J_f1, J_f2, J_f3

def solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma_1, gamma_2, alpha_14, alpha_41, dG_1=2, dG_2=2, k_bT=1):
    '''
    Solves for the steady-state probabilities of each state in the 6-state system.
    '''
    epsilon_1, epsilon_2, alpha_21, alpha_32, gamma_1, gamma_2, alpha_14, alpha_41, dG_1, dG_2, k_bT = mpmath.mpf(epsilon_1), mpmath.mpf(epsilon_2), \
          mpmath.mpf(alpha_21), mpmath.mpf(alpha_32),\
              mpmath.mpf(gamma_1), mpmath.mpf(gamma_2), \
                mpmath.mpf(alpha_14), mpmath.mpf(alpha_41), \
                    mpmath.mpf(dG_1), mpmath.mpf(dG_2), \
                        mpmath.mpf(k_bT)
    
    alpha_12 = alpha_21 * mpmath.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * mpmath.exp((dG_2 - epsilon_1)/ k_bT)

    alpha_54 = alpha_21
    alpha_65 = alpha_32
    alpha_45 = alpha_54 * mpmath.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * mpmath.exp((dG_2 - epsilon_2) / k_bT)



    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41

    a = np.array([[-(alpha_12 + alpha_14), alpha_21, gamma_1, alpha_41, 0 , 0],
                [alpha_12, -(alpha_21 + alpha_23 + alpha_25), alpha_32, 0, alpha_52, 0],
                [0, alpha_23, -(gamma_1 + alpha_32 + alpha_36), 0, 0, alpha_63],
                [alpha_14, 0, 0, -(alpha_41 + alpha_45), alpha_54, gamma_2],
                [0, alpha_25, 0, alpha_45, -(alpha_52 + alpha_54 + alpha_56), alpha_65],
                [0, 0, alpha_36, 0, alpha_56, -(gamma_2 + alpha_63 + alpha_65)]], dtype=float)  
    a[-1:,] = mpmath.mpf(1)
    bnp = np.zeros(a.shape[0])
    bnp[-1] = mpmath.mpf(1)

    a = mpmath.matrix(a.tolist())
    b = bnp.tolist()
    sol = np.array(mpmath.lu_solve(a,b)).astype(float)
    return sol

In [30]:
from bokeh.models import LinearColorMapper, ColorBar, BasicTicker
from bokeh.palettes import Spectral6
oscMidpoint = 0
k_bT = 1
dG_1 = 2
dG_2 = 2


alpha_21 = 1
alpha_32 = 1

alpha_54 = 1 
alpha_65 = 1 


## S1 definition
s1 = figure(title="Flux Diagram", x_range=(-1.1,1.1), y_range=(-1.1,1.1), tools="hover", tooltips="index: @name", toolbar_location=None, width=208, height=208)
s1.axis.visible = False
N = 6
node_indices = list(range(N))
graph = GraphRenderer()

graph.node_renderer.data_source.add(np.array(node_indices), 'index')
graph.node_renderer.data_source.add(np.array(node_indices) + 1, 'name')
graph.node_renderer.data_source.add(Spectral6, 'color')
graph.node_renderer.glyph = Circle(radius=0.1, fill_color="color")

# s = [0, 2, 1, 1, 2, 3, 1, 2, 3, 3, 0, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 3, 0, 0, 1, 1, 2, 3, 4, 4, 5, 5, 0, 4, 2, 3, 4]
# s = [0, 2, 1, 1, 1, 2, 3, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 0, 1, 1, 4, 4, 5, 5, 0, 4, 2, 3, 4]

s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]

graph.edge_renderer.data_source.data = dict(
    start=s,
    end=e)

### start of layout code
x = [-0.5, 0, 0.5, -0.5, 0, 0.5]
y = [0.5, 0.5, 0.5, -0.5, -0.5, -0.5]
graph_layout = dict(zip(range(len(node_indices)), zip(x, y)))
graph.layout_provider = StaticLayoutProvider(graph_layout=graph_layout)


### Draw quadratic bezier paths
def bezier(start, end, control, steps):
    return [(1-s)**2*start + 2*(1-s)*s*control + s**2*end for s in steps]
vSize = 6
xs, ys = [], []
steps = [i/100. for i in range(100)]
for edge_index in range(len(s)):
    sx, sy = graph_layout[s[edge_index]]
    ex, ey = graph_layout[e[edge_index]]
    match edge_index:
        # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
        # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
        case 0:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
    
        case 1: 
            x = bezier(sx, ex, -0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 2:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.75, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 3:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 4:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 5:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 6:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 7:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 8:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 9: 
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.25, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 10:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 11:
            x = bezier(sx, ex, 0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 12:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.75, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 13:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 14:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, 1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 15:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, -1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        
        case _:
            xs.append(np.linspace(sx, ex, 100))
            ys.append(np.linspace(sy, ey, 100))
    

  
    
graph.edge_renderer.data_source.data['xs'] = xs
graph.edge_renderer.data_source.data['ys'] = ys



s1.renderers.append(graph)

## S2 definition
probabilities = [r'1', r'2', r'3', r'4', r'5', r'6']
s2 = figure(title="State probabilities", x_range=probabilities, y_range=(0,1.0), tools="", toolbar_location=None, width=208, height=208)
source_1 = bokeh.models.ColumnDataSource(data=dict(x=probabilities, top=[0,0,0,0, 0, 0], color=Spectral6))

s2.vbar(x='x', top='top', width=0.9, source=source_1, color='color')

## S3 definition
cats = [ r'steady-state', r'dynamic steady-state', r'futile cycles']
fluxes = [r'nonenhanced', r'enhanced',  r'impeded', r'futile', r'optimum']
s3 = figure(title="Cycle fluxes", x_range=cats, y_range=(0,15), toolbar_location=None, tools="hover", tooltips="$name: @$name", width=288, height=208)
data = {'cats': cats,
        'optimum': [None,0, None],
        'nonenhanced': [0,0, None],
        'enhanced': [0,0, None],
        'impeded': [0,0, None],
        'futile': [None, None, 0],
        }
source_2 = bokeh.models.ColumnDataSource(data=data)
s3.vbar_stack(fluxes, x='cats', width=0.9, source=source_2, color=Spectral6[:-1], line_color=None, legend_label=fluxes)
s3.legend.location = "top_left"
s3.legend.orientation = "horizontal"
s3.yaxis.axis_label = "Flux [$$s^{-1}$$]"
s3.xaxis.major_label_text_font_size = '6pt'
s3.legend.label_text_font_size = '6pt'
s3.legend.items[3].visible = False
s3.legend.items[4].visible = False


 
def update(gamma=10, dG_1=2, dG_2=2, amplitude=2, alpha_41=25, alpha_14=25):
    #s1 update
    J_tot_list = find_J_tot_list(epsilon_list, dG_1, dG_2, alpha_21, alpha_32, gamma)
    idx_eps = np.argmin(np.abs(J_tot_list - np.max(J_tot_list)))
    oscMidpoint = epsilon_list[idx_eps] / 2

    epsilon_1 = float(oscMidpoint + amplitude/2)
    epsilon_2 = float(oscMidpoint - amplitude/2)
    
    # alpha_41 = alpha_14
    p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    p1 = float(p1)
    p2 = float(p2)
    p3 = float(p3)
    p4 = float(p4)
    alpha_12 = alpha_21 * np.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * np.exp((dG_2 - epsilon_1)/ k_bT)


    alpha_45 = alpha_54 * np.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * np.exp((dG_2 - epsilon_2) / k_bT)

    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41


    # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
    # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
    J_12 = np.max([p1 * alpha_12, 0.01])
    J_21 = np.max([p2 * alpha_21, 0.01])

    J_23 = np.max([p2 * alpha_23, 0.01])
    J_32 = np.max([p3 * alpha_32, 0.01])

    J_45 = np.max([p4 * alpha_45, 0.01])
    J_54 = np.max([p5 * alpha_54, 0.01])

    J_56 = np.max([p5 * alpha_56, 0.01])
    J_65 = np.max([p6 * alpha_65, 0.01])

    J_41 = np.max([p4 * alpha_41, 0.01])
    J_14 = np.max([p1 * alpha_14, 0.01])

    J_52 = np.max([p5 * alpha_52, 0.01])
    J_25 = np.max([p2 * alpha_25, 0.01])

    J_36 = np.max([p3 * alpha_36, 0.01])
    J_63 = np.max([p6 * alpha_63, 0.01])

    J_31 = np.max([p3 * gamma, 0.01])
    J_64 = np.max([p6 * gamma, 0.01])




    graph.edge_renderer.data_source.data["line_width"] = np.array([J_12, J_14, J_23, J_25, J_36, J_45, J_56, J_21, J_41, J_32, J_52, J_63, J_54, J_65, J_31, J_64])
    # graph.edge_renderer.data_source.data["line_width"] = [1] * len(s)
    graph.edge_renderer.glyph.line_width = {'field': 'line_width'}
    # S2 update 
    source_1.data = dict(x=probabilities, top=[p1, p2, p3, p4, p5, p6], color=Spectral6)

    # S3 update
    J_1, J_2, J_3, J_4, J_5, J_6, J_7, J_8, J_9, J_10, J_11, J_f1, J_f2, J_f3 = dynamic_steady_state_diagram(epsilon_1, epsilon_2, alpha_21, alpha_32, \
                                                                    gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    J_1 = float(J_1)
    J_2 = float(J_2)
    J_3 = float(J_3)
    J_4 = float(J_4)
    J_5 = float(J_5)
    J_6 = float(J_6)
    J_7 = float(J_7)
    J_8 = float(J_8)
    J_9 = float(J_9)
    J_10 = float(J_10)
    J_11 = float(J_11)
    J_f1 = float(J_f1)
    J_f2 = float(J_f2)
    J_f3 = float(J_f3)

    J_enhanced = np.abs(J_10) + np.abs(J_8)
    J_nonenhanced = np.abs(J_4) + np.abs(J_5) + np.abs(J_6) + np.abs(J_7)
    J_impeded = np.abs(J_9) + np.abs(J_11)
    J_futile = np.abs(J_1) + np.abs(J_2) + np.abs(J_3) + np.abs(J_f1) + np.abs(J_f2) + np.abs(J_f3)
    J_steady_state_optimum = np.max(J_tot_list)
    
    source_2.data = {'cats': cats,
        'optimum': [J_steady_state_optimum, None, None],
        'enhanced': [None, J_enhanced, None],
        'nonenhanced': [None, J_nonenhanced, None],
        'impeded': [None, J_impeded, None],
        'futile': [None,None, J_futile],
        }

s1.output_backend = "svg"
s2.output_backend = "svg"
s3.output_backend = "svg"
# plot = row(s3, )
plot = row(s1, s2, s3)
interact(update, gamma=(0, 20, 0.1,), dG_1 = (-5,5,0.1), dG_2 = (-5,5,0.1), amplitude=(0,5,0.1), alpha_41=(1e-11,100,1), alpha_14=(1e-11,100,1))

display(BokehModel(plot))

# export_svg(plot, filename="dynamic_steady_state_barplot.svg")

interactive_python_plots = True
output_notebook()

interactive(children=(FloatSlider(value=10.0, description='gamma', max=20.0), FloatSlider(value=2.0, descripti…

BokehModel(render_bundle={'docs_json': {'f6c93127-e823-4cff-af65-8f1ea1779b92': {'version': '3.6.2', 'title': …

Loading BokehJS ...

# Sankey diagram

In [ ]:
from bokeh.models import LinearColorMapper, ColorBar, BasicTicker
from bokeh.palettes import Spectral6
import holoviews as hv
from holoviews import dim, opts

hv.extension('bokeh')


oscMidpoint = 0
k_bT = 1
dG_1 = 2
dG_2 = 2


alpha_21 = 1
alpha_32 = 1

alpha_54 = 1 
alpha_65 = 1 


## S1 definition
s1 = figure(title="Flux Diagram", x_range=(-1.1,1.1), y_range=(-1.1,1.1), tools="hover", tooltips="index: @name", toolbar_location=None, width=208, height=208)
s1.axis.visible = False
N = 6
node_indices = list(range(N))
graph = GraphRenderer()

graph.node_renderer.data_source.add(np.array(node_indices), 'index')
graph.node_renderer.data_source.add(np.array(node_indices) + 1, 'name')
graph.node_renderer.data_source.add(Spectral6, 'color')
graph.node_renderer.glyph = Circle(radius=0.1, fill_color="color")

# s = [0, 2, 1, 1, 2, 3, 1, 2, 3, 3, 0, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 3, 0, 0, 1, 1, 2, 3, 4, 4, 5, 5, 0, 4, 2, 3, 4]
# s = [0, 2, 1, 1, 1, 2, 3, 0, 3, 2, 3, 4, 3, 5, 5, 5]
# e = [1, 0, 2, 3, 0, 1, 1, 4, 4, 5, 5, 0, 4, 2, 3, 4]

s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]

graph.edge_renderer.data_source.data = dict(
    start=s,
    end=e)

### start of layout code
x = [-0.5, 0, 0.5, -0.5, 0, 0.5]
y = [0.5, 0.5, 0.5, -0.5, -0.5, -0.5]
graph_layout = dict(zip(range(len(node_indices)), zip(x, y)))
graph.layout_provider = StaticLayoutProvider(graph_layout=graph_layout)


### Draw quadratic bezier paths
def bezier(start, end, control, steps):
    return [(1-s)**2*start + 2*(1-s)*s*control + s**2*end for s in steps]
vSize = 6
xs, ys = [], []
steps = [i/100. for i in range(100)]
for edge_index in range(len(s)):
    sx, sy = graph_layout[s[edge_index]]
    ex, ey = graph_layout[e[edge_index]]
    match edge_index:
        # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
        # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
        case 0:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
    
        case 1: 
            x = bezier(sx, ex, -0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 2:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.75, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 3:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 4:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 5:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 6:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 7:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0.25, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 8:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 9: 
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0.25, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 10:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, 0, steps)   
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 11:
            x = bezier(sx, ex, 0.75, steps)
            y = bezier(sy, ey, 0, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-5], y_start=y[-5], x_end=x[-4], y_end=y[-4], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 12:
            x = bezier(sx, ex, -0.25, steps)
            y = bezier(sy, ey, -0.75, steps)    
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-7], y_start=y[-7], x_end=x[-6], y_end=y[-6], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 13:
            x = bezier(sx, ex, 0.25, steps)
            y = bezier(sy, ey, -0.75, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-8], y_start=y[-8], x_end=x[-7], y_end=y[-7], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 14:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, 1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        case 15:
            x = bezier(sx, ex, 0, steps)
            y = bezier(sy, ey, -1.5, steps)
            xs.append(x)
            ys.append(y)
            arrow = Arrow(x_start=x[-4], y_start=y[-4], x_end=x[-3], y_end=y[-3], end=VeeHead(size=vSize))
            s1.add_layout(arrow)
        
        case _:
            xs.append(np.linspace(sx, ex, 100))
            ys.append(np.linspace(sy, ey, 100))
    

  
    
graph.edge_renderer.data_source.data['xs'] = xs
graph.edge_renderer.data_source.data['ys'] = ys



s1.renderers.append(graph)

## S2 definition
probabilities = [r'1', r'2', r'3', r'4', r'5', r'6']
s2 = figure(title="State probabilities", x_range=probabilities, y_range=(0,1.0), tools="", toolbar_location=None, width=208, height=208)
source_1 = bokeh.models.ColumnDataSource(data=dict(x=probabilities, top=[0,0,0,0, 0, 0], color=Spectral6))

s2.vbar(x='x', top='top', width=0.9, source=source_1, color='color')

## S3 definition
cats = [ r'steady-state', r'dynamic steady-state', r'futile cycles']
fluxes = [r'nonenhanced', r'enhanced',  r'impeded', r'futile', r'optimum']
s3 = figure(title="Cycle fluxes", x_range=cats, y_range=(0,15), toolbar_location=None, tools="hover", tooltips="$name: @$name", width=288, height=208)
data = {'cats': cats,
        'optimum': [None,0, None],
        'nonenhanced': [0,0, None],
        'enhanced': [0,0, None],
        'impeded': [0,0, None],
        'futile': [None, None, 0],
        }
source_2 = bokeh.models.ColumnDataSource(data=data)
s3.vbar_stack(fluxes, x='cats', width=0.9, source=source_2, color=Spectral6[:-1], line_color=None, legend_label=fluxes)
s3.legend.location = "top_left"
s3.legend.orientation = "horizontal"
s3.yaxis.axis_label = "Flux [$$s^{-1}$$]"
s3.xaxis.major_label_text_font_size = '6pt'
s3.legend.label_text_font_size = '6pt'
s3.legend.items[3].visible = False
s3.legend.items[4].visible = False


 
def update(gamma=10, dG_1=2, dG_2=2, amplitude=2, alpha_41=25, alpha_14=25):
    #s1 update
    J_tot_list = find_J_tot_list(epsilon_list, dG_1, dG_2, alpha_21, alpha_32, gamma)
    idx_eps = np.argmin(np.abs(J_tot_list - np.max(J_tot_list)))
    oscMidpoint = epsilon_list[idx_eps] / 2

    epsilon_1 = float(oscMidpoint + amplitude/2)
    epsilon_2 = float(oscMidpoint - amplitude/2)
    
    # alpha_41 = alpha_14
    p1, p2, p3, p4, p5, p6 = solve_p_dynamic_steady_state(epsilon_1, epsilon_2, alpha_21, alpha_32, gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    p1 = float(p1)
    p2 = float(p2)
    p3 = float(p3)
    p4 = float(p4)
    alpha_12 = alpha_21 * np.exp((dG_1 + epsilon_1) / k_bT)
    alpha_23 = alpha_32 * np.exp((dG_2 - epsilon_1)/ k_bT)


    alpha_45 = alpha_54 * np.exp((dG_1 + epsilon_2) / k_bT)
    alpha_56 = alpha_65 * np.exp((dG_2 - epsilon_2) / k_bT)

    alpha_25 = alpha_14
    alpha_52 = alpha_41
    alpha_36 = alpha_14
    alpha_63 = alpha_41


    # s = [0, 0, 1, 1, 2, 3, 4, 1, 3, 2, 4, 5, 4, 5, 2, 5]
    # e = [1, 3, 2, 4, 5, 4, 5, 0, 0, 1, 1, 2, 3, 4, 0, 3]
    J_12 = np.max([p1 * alpha_12, 0.01])
    J_21 = np.max([p2 * alpha_21, 0.01])

    J_23 = np.max([p2 * alpha_23, 0.01])
    J_32 = np.max([p3 * alpha_32, 0.01])

    J_45 = np.max([p4 * alpha_45, 0.01])
    J_54 = np.max([p5 * alpha_54, 0.01])

    J_56 = np.max([p5 * alpha_56, 0.01])
    J_65 = np.max([p6 * alpha_65, 0.01])

    J_41 = np.max([p4 * alpha_41, 0.01])
    J_14 = np.max([p1 * alpha_14, 0.01])

    J_52 = np.max([p5 * alpha_52, 0.01])
    J_25 = np.max([p2 * alpha_25, 0.01])

    J_36 = np.max([p3 * alpha_36, 0.01])
    J_63 = np.max([p6 * alpha_63, 0.01])

    J_31 = np.max([p3 * gamma, 0.01])
    J_64 = np.max([p6 * gamma, 0.01])




    graph.edge_renderer.data_source.data["line_width"] = np.array([J_12, J_14, J_23, J_25, J_36, J_45, J_56, J_21, J_41, J_32, J_52, J_63, J_54, J_65, J_31, J_64])
    # graph.edge_renderer.data_source.data["line_width"] = [1] * len(s)
    graph.edge_renderer.glyph.line_width = {'field': 'line_width'}
    # S2 update 
    source_1.data = dict(x=probabilities, top=[p1, p2, p3, p4, p5, p6], color=Spectral6)

    # S3 update
    J_1, J_2, J_3, J_4, J_5, J_6, J_7, J_8, J_9, J_10, J_11, J_f1, J_f2, J_f3 = dynamic_steady_state_diagram(epsilon_1, epsilon_2, alpha_21, alpha_32, \
                                                                    gamma, gamma, alpha_14, alpha_41, dG_1=dG_1, dG_2=dG_2, k_bT=1)
    J_1 = float(J_1)
    J_2 = float(J_2)
    J_3 = float(J_3)
    J_4 = float(J_4)
    J_5 = float(J_5)
    J_6 = float(J_6)
    J_7 = float(J_7)
    J_8 = float(J_8)
    J_9 = float(J_9)
    J_10 = float(J_10)
    J_11 = float(J_11)
    J_f1 = float(J_f1)
    J_f2 = float(J_f2)
    J_f3 = float(J_f3)

    J_enhanced = np.abs(J_10) + np.abs(J_8)
    J_nonenhanced = np.abs(J_4) + np.abs(J_5) + np.abs(J_6) + np.abs(J_7)
    J_impeded = np.abs(J_9) + np.abs(J_11)
    J_futile = np.abs(J_1) + np.abs(J_2) + np.abs(J_3) + np.abs(J_f1) + np.abs(J_f2) + np.abs(J_f3)
    J_steady_state_optimum = np.max(J_tot_list)
    
    source_2.data = {'cats': cats,
        'optimum': [J_steady_state_optimum, None, None],
        'enhanced': [None, J_enhanced, None],
        'nonenhanced': [None, J_nonenhanced, None],
        'impeded': [None, J_impeded, None],
        'futile': [None,None, J_futile],
        }

s1.output_backend = "svg"
s2.output_backend = "svg"
s3.output_backend = "svg"
# plot = row(s3, )
plot = row(s1, s2, s3)
interact(update, gamma=(0, 20, 0.1,), dG_1 = (-5,5,0.1), dG_2 = (-5,5,0.1), amplitude=(0,5,0.1), alpha_41=(1e-11,100,1), alpha_14=(1e-11,100,1))

display(BokehModel(plot))

# export_svg(plot, filename="dynamic_steady_state_barplot.svg")

interactive_python_plots = True
output_notebook()

interactive(children=(FloatSlider(value=10.0, description='gamma', max=20.0), FloatSlider(value=2.0, descripti…

BokehModel(render_bundle={'docs_json': {'2aa2e6b2-b2c3-459f-b7da-fc90c4b72b69': {'version': '3.6.2', 'title': …

Loading BokehJS ...